# Causal pretrained encoders: anticipate code switches and their duration

Fine-tunes a **pretrained encoder with a causal attention mask** on sub-word labels. At every piece, the model sees only the pieces so far and predicts:
- **`y_sw`:** does the next piece switch language?
- **`y_dur`:** if so, is the new run Small (1–2), Medium (3–6) or Large (7+)?

| `MODEL` | Checkpoint | Pretrained on | Tokenizer |
|---|---|---|---|
| `"xlmr"` | xlm-roberta-base (278 M) | 100 languages | SentencePiece |
| `"roberta"` | roberta-base (125 M) | English only (the monolingual control) | byte-level BPE |
| `"mbert"` | bert-base-multilingual-cased (178 M) | 104 languages | WordPiece |

The protocol is the same as every other model in the project:

| Split | Share | Used for |
|---|---|---|
| train | 80% | fitting (`row_id % 10` in 0–7) |
| val | 10% | tuning the switch threshold (8) |
| test | 10% | the final numbers, scored once (9) |

The report (`src/evaluation.py`) covers switch P/R/F1 overall, per language pair, macro-F1 across pairs, per direction, and 3-class duration. It also includes **shared positions**, the last piece of each non-CJK word. Those are the same words for every tokenizer, so that score compares these models fairly. A causality check runs before training and must report `PREFIX-ONLY OK`.

**Short vs Large duration.** `DUR_CLASSES = 2` trains a two-class head: Short (1–6) vs Large (7+). It needs no new labels, because Small and Medium are merged on the fly (`anticipation.collapse_duration`). The final binary runs (2026-11-20) used: `STEPS = 36_000`, `PREV_LANG = True`, `DECIDE_AT = "gap"`, `RESHUFFLE = True`, `EVAL_EVERY = 3000`, `DUR_CLASSES = 2`, `SAVE_CHECKPOINT = True`, `SUFFIX = "v2"`. That gives the tag `<model>_bin_prev_gap_36k_rsv2`, which is directly comparable to `<model>_prev_gap_36k_rsv2` (3-class). Expect about 3.6 h on a T4 for XLM-R.

**Multi-dataset training (SwitchLingua + LinCE).** `MIX_LINCE = w` (an integer ≥ 1) trains on SwitchLingua plus the LinCE **train** tweets (`src/lince.mix`), with LinCE's training-split rows repeated `w` times. LinCE **validation** is never trained on: it is the held-out LinCE test. τ is still tuned on SwitchLingua val, and test is scored on the SwitchLingua test split (`--eval-path`), the same rows as every other run. The LinCE cell then scores LinCE validation only. The tag gets `_lince<w>`. The 2026-11 runs use the final-run flags with `MIX_LINCE = 1` and `5`, which put LinCE at about 3% and 12% of training pieces (10% and 36% of training windows). LinCE downloads from its public Hugging Face mirror, so nothing needs uploading.

## How to run

1. **Settings:** set the Accelerator to **GPU T4 ×2** (one GPU is used) and turn Internet **on**.
2. **Secret:** Add-ons → Secrets → `HF_TOKEN`, a read token with access to both `CodeSwitchGroup4/switchlingua-tagged-final` (our labels; a fine-grained token needs the `CodeSwitchGroup4` org added under its permissions) and the gated `Shelton1013/SwitchLingua_text` (the original text).
3. **Settings cell:** set `MODEL`, then **Save Version → Save & Run All**, so the run continues if you close the browser.
   - Use `SMOKE = True` first for a quick check.
   - A full run of 12,000 steps took about 72 minutes on a T4 for XLM-R; RoBERTa and mBERT are smaller.
4. **Collect the results:** from the version's Output tab, download `data/results/anticipation_<TAG>.json` (and `_zeroshot.json`, if run), `data/preds/<TAG>.npz`, `data/results/lince_<TAG>.json` and, for the replay dashboard's live mode, `models/anticipation_<TAG>.pt` (about 1.1 GB for XLM-R), into the same paths in the repo. `TAG` is the model name plus the tuning flags that are on (e.g. `xlmr_durw_prev`). Then run `./.venv/bin/python scripts/analysis/compare_models.py`. The saved predictions let thresholds be tuned locally with no GPU.

## Config

In [ ]:
MODEL = "xlmr"          # "xlmr" | "roberta" | "mbert"
SMOKE = False           # True: 60 steps + a small evaluation, to check everything runs
RUN_FULL = True         # train and score on all 15 pairs
ZERO_SHOT = False       # train on 10 pairs and score the 4 held-out languages
STEPS = 12_000          # 1 epoch = 11,711 steps on all 15 pairs at batch 16 (8,049 on the 6 script pairs)
BATCH = 16
WINDOW, STRIDE = 384, 288   # these encoders read at most 512 positions
LR, HEAD_LR = 2e-5, 1e-3    # pretrained body vs. the two new heads
EVAL_BATCHES = 0        # 0 = score the whole val and test splits
RESHUFFLE = False       # True: a new data order every epoch (for multi-epoch STEPS; changes 12k-step runs slightly)
EVAL_EVERY = 0          # >0: score a fixed val slice every N steps (learning curve in the json), e.g. 3000
# Tuning flags (all off = the baseline setup). See README "How models are evaluated".
SEED = 0                # reproducible initialisation and data order
DUR_WEIGHTS = "none"    # "balanced": class-weighted duration loss (weights from the train split)
DUR_WEIGHT = 0.3        # lambda_2 in L = L_sw + lambda_2 * L_dur (0.3 = every run before 2026-11-10)
DUR_HEAD = "softmax"    # "ordinal": cumulative cuts P(>Small), P(>Medium) (duration is ordered)
DUR_CLASSES = 3         # 2: Short (1-6) vs Large (7+), --dur-classes 2 (needs DUR_HEAD "softmax"); labels collapse on the fly
TURN_SEP = False        # separator piece between speaker turns
PREV_LANG = False       # input: the language of the last finished word
PAIR_EMB = False        # input: a learned embedding of the row's language pair (15 pairs)
LORA_R = 0              # >0: freeze the encoder, train rank-r LoRA adapters (16 is typical)
LORA_LR = 2e-4          # learning rate of the adapters (replaces LR when LORA_R > 0)
BACKBONE = ""           # "" = the MODEL's default checkpoint; e.g. "xlm-roberta-large" (same tokenizer)
PREDICT_AT = "piece"    # "word_end": train and score only at the last piece of each word
DECIDE_AT = "gap"       # adopted 2026-11-03 (groupUpdate2 §1): each label sits on the punctuation after its word; "word" = Update #1
CONTEXT = None          # 1: read the current piece only, no prefix (needs the three flags above off)
PAIRS = ""              # comma-separated first languages to train and score on ("" = all 15),
                        # e.g. "Cantonese,Chinese,Japanese,Korean,Russian,Arabic"
SAVE_CHECKPOINT = True  # 0.5-1.1 GB in /kaggle/working; the final weights, needed for the replay dashboard's live mode
EVAL_LINCE = True       # after training, score the checkpoint on LinCE tweets (human tags; spaeng + hineng)
MIX_LINCE = 0           # >0: also train on LinCE train tweets, repeated this many times (LinCE validation stays held out)
SUFFIX = ""             # appended to TAG, e.g. "_v2" so a rerun on new labels does not overwrite an old run's files
LABELS_REPO = "CodeSwitchGroup4/switchlingua-tagged-final"

## Setup

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "wordfreq"], check=True)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available():
    print("WARNING: no GPU - enable Settings > Accelerator > GPU T4 x2")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("xlmr_out")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
for d in ("src", "scripts/training", "data/results", "data/preds", "models", "logs"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if TOKEN:
    os.environ["HF_TOKEN"] = TOKEN       # so every Hub download is authenticated, the tokenizer's too
print("working dir:", WORK, "| HF token:", "found" if TOKEN else "MISSING - add the HF_TOKEN secret")

## Project code
The live `src/` modules and the training script, embedded verbatim.

In [ ]:
%%writefile src/__init__.py
# package marker

In [ ]:
%%writefile src/routing.py
"""Language-pair -> tagging-strategy lookup.

Two regimes, confirmed empirically against the cached SwitchLingua CSVs:

* ``unicode`` - first_language uses a script disjoint from Latin, so a token's
  script *is* its language. Exact, no model. ~68.7% of rows.
* ``model``   - first_language is Latin-script (or romanized), so only
  vocabulary separates it from the second language. Needs constrained LID.

Note the plan's routing table listed Arabic's pair as Arabic<->French; the data
says ``Arabic_eng.csv`` is Arabic<->English, and French appears as a *second*
language only in ``German_French.csv``. Nothing here hardcodes the second
language - callers pass the row's own ``second_language``.
"""

SCRIPT_DISJOINT_CONFIRMED = {
    "Cantonese", "Chinese", "Japanese", "Korean", "Russian", "Arabic",
}

# Resolved from the plan's `unicode_pending_check` gate: 54.7% of Hindi rows are
# romanized Hinglish ("Woh kal Prime Minister Carney se milne wali hain"), so
# Devanagari-vs-Latin is not a reliable split for this language. Routed to model.
NEEDS_MODEL = {
    "German", "Philippines", "Vietnamese", "Malay", "Spanish", "French",
    "Italian", "Hindi",
}

# fastText lid.176 label codes.
LANG_TO_CODE = {
    "German": "de", "Philippines": "tl", "Vietnamese": "vi", "Malay": "ms",
    "Spanish": "es", "French": "fr", "Italian": "it", "English": "en",
    "Hindi": "hi", "Russian": "ru", "Arabic": "ar", "Chinese": "zh",
    "Cantonese": "zh", "Japanese": "ja", "Korean": "ko",
}

# Script expected for each unicode-routed first_language. A token in one of
# these scripts is the first language; a Latin token is the second language.
_EXPECTED_SCRIPT = {
    "Cantonese": {"Han"},
    "Chinese": {"Han"},
    "Japanese": {"Han", "Hiragana", "Katakana"},
    "Korean": {"Hangul"},
    "Russian": {"Cyrillic"},
    "Arabic": {"Arabic"},
    "Hindi": {"Devanagari"},
}

# fastText's `hi` model is trained on Devanagari. Romanized Hindi tokens are
# out of distribution for every label, so rows here are flagged low-confidence
# rather than silently trusted.
ROMANIZATION_RISK = {"Hindi"}


# --- tier 3: context-aware LinCE BERT token classifiers ---------------------
#
# Measured on hand-labelled tokens from the real rows (NE excluded), fastText
# vs. the spaeng model used cross-lingually:
#
#     language      fastText   spaeng    -> routed to
#     German           100%      80%        fasttext
#     French            90%      68%        fasttext
#     Vietnamese        90%      88%        fasttext   (tie; diacritics carry it)
#     Italian           96%     100%        spaeng
#     Spanish           94%     100%        spaeng
#     Philippines       86%      94%        spaeng
#     Malay             75%      88%        spaeng
#     Hindi          broken       -         hineng
#
# spaeng transfers as a generic Latin-L1-vs-English discriminator to Romance and
# Austronesian languages but *regresses* Germanic/French, so routing is per
# language and empirical, not by family. hineng does not transfer at all (17%
# Malay / 64% Tagalog) - it is Hindi-only.
#
# Sample sizes are small (n=26-48 per language); treat Vietnamese and Italian,
# where the gap is inside a couple of tokens, as ties rather than firm results.
MODEL_REPO = {
    "spaeng": "sagorsarker/codeswitch-spaeng-lid-lince",
    "hineng": "sagorsarker/codeswitch-hineng-lid-lince",
}
MODEL_L1_LABEL = {"spaeng": "spa", "hineng": "hin"}
MODEL_FOR = {
    "Hindi": "hineng",
    "Spanish": "spaeng", "Italian": "spaeng",
    "Malay": "spaeng", "Philippines": "spaeng",
}


# --- tier 4: dictionary frequency comparison (measured best) ----------------
#
# Measured on data/gold_eval.json (324 hand-labelled tokens) and confirmed on
# 767 rows against the dataset's own cs_ratio_result:
#
#   language      dict-freq   best model      ratio err (dict vs prod)
#   Malay            100%      88% mBERT        0.103 vs 0.183
#   Philippines      100%      94% mBERT        0.147 vs 0.222
#   French           100%      90% fastText     0.121 vs 0.153
#   German           100%     100% fastText     0.133 vs 0.151
#   Vietnamese       100%      90% fastText     0.084 vs 0.093
#   Spanish          100%     100% mBERT        0.105 vs 0.104
#   Italian          100%     100% mBERT        0.132 vs 0.124
#   Hindi (roman)     66%     100% mBERT        - model required
#
# dict-freq wins or ties on every Latin-script pair and needs no model at all.
# It also handles a non-English second language (the German<->French rows),
# which neither LinCE model can, so fastText and mBERT-spaeng both retire.
#
# Romanized Hindi is the sole exception: wordfreq's `hi` list is Devanagari and
# romanized Hinglish has no standard orthography, so no word list can cover it.
# Four independent techniques agree the context-aware models are required there
# (fastText 56%, dict-freq 66%, mBERT 100%, XLM-R 100%, CANINE 98%).
USES_DICT = {
    "Spanish", "Italian", "Malay", "Philippines",
    "German", "French", "Vietnamese",
}




# --- tier 5: AnE-LID (current production for every "<Latin L1> vs English") --
#
# The gold set above saturated (every tier at ~99-100% on 26-59 self-labelled
# tokens), so the decision was made on human-annotated public data instead
# (scripts/labelling/eval_public.py, L1/EN tokens only):
#
#   set                      n     dict-freq   mBERT    AnE-LID
#   LinCE spaeng dev      31,667     94.1%     98.8%     99.4%
#   LinCE hineng dev      12,303     80.5%     97.4%     98.8%
#   Denglisch (de-en)     59,485     96.2%     53.0%     99.9%  (likely seen in training)
#
# dict-freq's "100%" on the gold set was ~94-96% on human labels. AnE-LID is
# multi-pair trained, so it also covers romanized Hindi and replaces mBERT
# hineng there. It only answers "English or not", so German->French keeps
# dict-freq. Where AnE and dict-freq agree, tokens are 99.3-100% correct.
ANE_L2 = "English"


def ane_to_language(label: str, first_language: str, second_language: str = "English") -> str:
    """Map a raw AnE-LID label to the row's language. ``Mixed`` (an English stem
    in L1 morphology, "rewatchen") goes to L1, its grammatical host."""
    if label == "English":
        return second_language
    if label in ("notEnglish", "Mixed"):
        return first_language
    return "other"


def route(first_language: str, second_language: str = "English") -> str:
    """Return ``"unicode"``, ``"ane"`` or ``"dict"`` for a row.

    Hindi routes per *row* rather than per language - see
    ``dispatch._hindi_is_devanagari``. Devanagari rows are script-disjoint from
    English and go to the exact tagger; only romanized rows need the model.
    ``"bert"`` (mBERT hineng) is retired but still dispatchable for comparison.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "unicode"
    if second_language != ANE_L2:
        return "dict"
    return "ane"


def lang_code(language: str) -> str | None:
    """Map a dataset language name to its fastText lid.176 label code."""
    return LANG_TO_CODE.get(language)


def expected_script(language: str) -> set[str]:
    """Unicode script name(s) that indicate ``language`` in a mixed token stream."""
    return _EXPECTED_SCRIPT.get(language, {"Latin"})


# --- V1 quality tiers (measured, not assumed) -------------------------------
#
# The plan's escalation rule needs a per-pair accuracy number. Manual labelling
# was not required to get one: the six script-disjoint languages are tagged
# *exactly*, so their disagreement with the dataset's own `computed_ratio`
# (-0.134 .. +0.117, mean |diff| ~0.075) measures the noise in that reference
# signal rather than any tagger error. That band is the yardstick - a
# model-routed pair sitting inside it is behaving like an exact tagger; one far
# outside it is genuinely broken.
#
# Measured on a 150-row-per-language stratified sample (see notebook cell):
#
#   PASS  (inside the exact-tagger band)   Vietnamese .056  Spanish .086
#                                          German .118  French .128  Italian .129
#   FAIL  (2-5x outside it)                Malay .277  Hindi .369  Philippines .406
#
# This inverts the plan's expectation, which predicted German/Malay/Tagalog
# would pass easily and the Romance pairs would fail on shared function words.
# The Romance pairs are fine; lid.176 simply has far less Malay/Tagalog training
# data, and romanized Hindi is not in its label space at all.
V1_PASSES = {"Vietnamese", "Spanish", "German", "French", "Italian"}
V1_FAILS_NEEDS_V2 = {"Malay", "Philippines", "Hindi"}


# Measured token accuracy after routing each language to its best tagger.
# `low` now means "still under the plan's 90% bar with the best option tried".
# Accuracy of the tagger each language is now routed to, on the gold set.
# Every Latin pair scored 100% under dict-freq - but on only 26-50 tokens each,
# self-labelled. `provisional` marks that: good enough to ship, not yet a
# number to publish. The review tool exists to replace these.
_MEASURED_ACCURACY = {
    "Spanish": 1.00, "Italian": 1.00, "Malay": 1.00, "Philippines": 1.00,
    "German": 1.00, "French": 1.00, "Vietnamese": 1.00,   # dict-freq
    "Hindi": 1.00,                                        # mBERT hineng
}


def tagger_quality(first_language: str, second_language: str = "English") -> str:
    """``exact`` | ``provisional`` - how far to trust a row's tags.

    ``exact`` is correct by construction (script-disjoint). ``provisional``
    means measured at >=90% on a small self-labelled sample; human review is
    what promotes it to a defensible number.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "exact"
    return "provisional" if _MEASURED_ACCURACY.get(first_language, 0) >= 0.90 else "low"

In [ ]:
%%writefile src/tokenizer.py
"""Mixed-granularity tokenizer.

CJK does not tokenize on whitespace - a run of Han/Kana/Hangul between two
English words may hold several words with no space - so those scripts are cut
per character. Everything else is cut per word.

Two fixes over the pattern in the plan:

1. ``[A-Za-z]+`` splits accented Latin: "dificil" became ``dif|i|cil``, which
   shatters every Spanish/French/German/Vietnamese word carrying a diacritic
   and would badly inflate run counts. ``\\p{Script=Latin}`` covers the accented
   range, since U+00E9 and friends *are* Latin script.
2. Cyrillic, Arabic and Devanagari are space-delimited, so word-level is right
   for them; the plan's fallback branch would have cut them per character.
"""
import regex as re

# Order matters: character-level CJK alternatives are tried before the
# word-level ones, and the single-character fallback is last.
_PATTERN = re.compile(
    r"\p{Script=Han}"
    r"|\p{Script=Hiragana}"
    r"|\p{Script=Katakana}"
    r"|\p{Script=Hangul}"
    r"|[\p{Script=Latin}\p{M}]+(?:['’][\p{Script=Latin}\p{M}]+)*"
    r"|[\p{Script=Cyrillic}\p{M}]+"
    r"|[\p{Script=Arabic}\p{M}]+"
    r"|[\p{Script=Devanagari}\p{M}]+"
    r"|\d+"
    r"|[^\s]"
)


def tokenize_mixed(text: str) -> list[str]:
    """Split text into CJK-character tokens and word tokens for other scripts."""
    return _PATTERN.findall(text)

In [ ]:
%%writefile src/dict_tagger.py
"""Dictionary-lookup tagging - the obvious approach, measured.

Two variants of "look the word up in each language's dictionary":

``tag_naive``      if the word exists in the second language's list, call it the
                   second language. This is the intuitive rule, and it fails on
                   lexical overlap: most *common* words exist in both lists.
``tag_frequency``  compare how common the word is in each language and take the
                   higher. Strictly better, and the reason is instructive - the
                   question is never "is this word in the dictionary" but "which
                   language is it more characteristic of".

Frequencies come from `wordfreq` (Zipf scale: ~7 = very common, 0 = absent).
"""
from __future__ import annotations

from functools import lru_cache

from wordfreq import zipf_frequency

LANG_TO_CODE = {
    "English": "en", "Spanish": "es", "German": "de", "French": "fr",
    "Italian": "it", "Malay": "ms", "Philippines": "fil",
    "Vietnamese": "vi", "Hindi": "hi",
}


@lru_cache(maxsize=500_000)
def zipf(word: str, code: str) -> float:
    return zipf_frequency(word, code)


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def tag_naive(tokens: list[str], first_language: str,
              second_language: str) -> list[tuple[str, str]]:
    """"If it's in the second language's dictionary, it's the second language.\""""
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t):
            out.append((t, "other"))
        elif zipf(t.lower(), l2) > 0:
            out.append((t, second_language))
        else:
            out.append((t, first_language))
    return out


def tag_frequency(tokens: list[str], first_language: str, second_language: str,
                  margin: float = 0.0) -> list[tuple[str, str]]:
    """Whichever language the word is more characteristic of wins."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t) or l1 is None:
            out.append((t, "other" if not _is_word(t) else first_language))
            continue
        w = t.lower()
        f1, f2 = zipf(w, l1), zipf(w, l2)
        if f1 == 0 and f2 == 0:
            out.append((t, "other"))          # unknown to both: proper noun etc.
        else:
            out.append((t, second_language if f2 > f1 + margin else first_language))
    return out


def overlap_stats(tokens: list[str], first_language: str,
                  second_language: str) -> dict:
    """How often is a word present in BOTH dictionaries? That is the ceiling
    on the naive rule: every such word is a coin flip for it."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    words = [t.lower() for t in tokens if _is_word(t)]
    if not words or l1 is None:
        return dict(words=0, both=0, neither=0, only_l1=0, only_l2=0)
    both = neither = only1 = only2 = 0
    for w in words:
        a, b = zipf(w, l1) > 0, zipf(w, l2) > 0
        both += a and b
        neither += not a and not b
        only1 += a and not b
        only2 += b and not a
    return dict(words=len(words), both=both, neither=neither,
                only_l1=only1, only_l2=only2)

In [ ]:
%%writefile src/lexical_override.py
"""Overrule AnE-LID where dict-freq's evidence is overwhelming (added 2026-11-03).

AnE-LID was chosen on LinCE spa-eng / hin-eng and Denglisch, the only
human-annotated sets we have. It was never checked on Malay, Vietnamese or
Tagalog, and there it tags common L1 function words English: Malay ``akan``
×1,496, ``dan``, ``yang``; Vietnamese ``và``, ``sẽ``, ``của``, even spans like
"bekerjasama dengan agensi lain untuk". Each one invents two switches.
``scripts/analysis/consensus_f1.py`` found 40-45% of the positions around such words
labelled as switches, and the trained model refusing them (recall 0.16-0.18).

Two rules, applied to word tokens that are not names:

1. **Vietnamese letters.** In a Vietnamese row, a word containing ``đ ă ơ ư``
   or a stacked tone mark (U+1EA0-U+1EF9) is Vietnamese. No English word has
   them; French/Spanish accents (é, ê, ñ) are not in the set.
2. **Frequency margin.** The word flips to the other language of the pair when
   it is common there (Zipf >= ``FLOOR``) and at least ``MARGIN`` Zipf more
   common there than in the language AnE gave it. Margin 2.0 = 100x more
   frequent. Shared words (``a``, ``in``, ``me``, ``online``) never clear it.
   All-caps tokens skip this rule: an acronym's lowercase form is an unrelated
   word ("AI" vs French "ai", as in j'ai).

Scope: ``OVERRIDE_L1`` with English as L2. Romanized Hindi is excluded
(wordfreq's Hindi list is Devanagari, so dict-freq can't read it), German->French
is dict-freq already, and script-tagged pairs are exact.
"""
from __future__ import annotations

import unicodedata

from .dict_tagger import LANG_TO_CODE, zipf

OVERRIDE_L1 = frozenset({"Spanish", "Italian", "Malay", "Philippines", "German", "French",
                         "Vietnamese"})
MARGIN = 2.0
FLOOR = 4.0
_VI_LETTERS = frozenset("đĐăĂơƠưƯ")


def has_vietnamese_letters(token: str) -> bool:
    """True if the word has a letter only Vietnamese uses (checked after NFC)."""
    for ch in unicodedata.normalize("NFC", token):
        if ch in _VI_LETTERS or 0x1EA0 <= ord(ch) <= 0x1EF9:
            return True
    return False


def in_scope(first_language: str, second_language: str) -> bool:
    return second_language == "English" and first_language in OVERRIDE_L1


def override_tagged(tagged: list[tuple[str, str]], first_language: str, second_language: str,
                    is_entity: list[bool] | None = None, margin: float = MARGIN,
                    floor: float = FLOOR) -> tuple[list[tuple[str, str]], list[bool]]:
    """Return ``(tagged, changed)``: the corrected labels and which tokens moved."""
    if not in_scope(first_language, second_language):
        return list(tagged), [False] * len(tagged)
    l1, l2 = first_language, second_language
    c1, c2 = LANG_TO_CODE[l1], LANG_TO_CODE[l2]
    ents = is_entity if is_entity is not None else [False] * len(tagged)
    out, changed = [], []
    for (tok, lang), ent in zip(tagged, ents):
        new = lang
        if not ent and lang in (l1, l2) and any(ch.isalpha() for ch in tok):
            if l1 == "Vietnamese" and has_vietnamese_letters(tok):
                new = l1
            elif not (len(tok) >= 2 and tok.isupper()):    # acronyms: "AI" is not French "ai"
                w = tok.lower()
                f1, f2 = zipf(w, c1), zipf(w, c2)
                if lang == l2 and f1 >= floor and f1 - f2 >= margin:
                    new = l1
                elif lang == l1 and f2 >= floor and f2 - f1 >= margin:
                    new = l2
        out.append((tok, new))
        changed.append(new != lang)
    return out, changed

In [ ]:
%%writefile src/unicode_tagger.py
"""Script-based tagging, for language pairs whose scripts are disjoint.

Exact by construction: if the first language is written in Han and the second
in Latin, a token's script settles which language it belongs to. No model, no
threshold, nothing to validate beyond the script table itself.
"""
import regex as re

from .routing import expected_script

_SCRIPTS = ["Han", "Hiragana", "Katakana", "Hangul", "Devanagari",
            "Cyrillic", "Arabic", "Latin"]
_SCRIPT_RE = {name: re.compile(fr"\p{{Script={name}}}") for name in _SCRIPTS}


def char_script(ch: str) -> str:
    for name in _SCRIPTS:
        if _SCRIPT_RE[name].match(ch):
            return name
    return "other"


def tag_token_unicode(token: str) -> str:
    """Return the token's single script name, or ``"other"`` if mixed/unscripted.

    Combining marks are ignored when deciding, so an accented Latin token stays
    ``Latin`` rather than falling through to ``other``.
    """
    scripts = {char_script(c) for c in token if not c.isspace()}
    scripts.discard("other")
    if len(scripts) == 1:
        return scripts.pop()
    return "other"


def tag_tokens_unicode(tokens: list[str], first_language: str,
                       second_language: str) -> list[tuple[str, str]]:
    """Tag tokens as the row's first or second language by script.

    Returns ``(token, language)`` pairs, where language is the row's own
    ``first_language`` / ``second_language`` value, or ``"other"`` for
    punctuation and digits.
    """
    l1_scripts = expected_script(first_language)
    l2_scripts = expected_script(second_language)
    out = []
    for tok in tokens:
        script = tag_token_unicode(tok)
        if script in l1_scripts:
            out.append((tok, first_language))
        elif script in l2_scripts:
            out.append((tok, second_language))
        else:
            out.append((tok, "other"))
    return out

In [ ]:
%%writefile src/dispatch.py
"""Single entry point that routes a token stream to the right tagger.

Kept separate so ``segments``, ``inspection`` and ``persist`` share one copy of
the branch, and so the heavy imports (torch, the models) stay lazy - a
unicode-routed row loads none of them.
"""
from __future__ import annotations

import regex as re

from .routing import ane_to_language, route

_DEVANAGARI = re.compile(r"\p{Script=Devanagari}")
_LATIN = re.compile(r"\p{Script=Latin}")


def _hindi_is_devanagari(tokens: list[str]) -> bool:
    """Hindi rows come in two scripts; only the romanized half needs a model.

    Routing per *language* sent Devanagari rows to the LinCE model, which is
    trained on romanized Hinglish and so is out of distribution for them - while
    the Unicode tagger handles Devanagari-vs-Latin exactly. Decide per row.
    """
    dev = sum(1 for t in tokens if _DEVANAGARI.search(t))
    lat = sum(1 for t in tokens if _LATIN.search(t))
    return dev > lat


def row_strategy(tokens: list[str], first_language: str, second_language: str) -> str:
    """``route`` plus the per-row Devanagari check for Hindi."""
    strategy = route(first_language, second_language)
    if (strategy in ("ane", "bert") and first_language == "Hindi"
            and _hindi_is_devanagari(tokens)):
        return "unicode"
    return strategy


def _devanagari_is_hindi(tagged, first_language):
    """Romanized-majority Hindi rows still carry some Devanagari words, and the
    LinCE model called 24,113 of them English. Script settles those exactly."""
    if first_language != "Hindi":
        return tagged
    return [(t, first_language if _DEVANAGARI.search(t) else lang) for t, lang in tagged]


def _names_from_context(tagged, is_entity, ane_labels, first_language, second_language):
    """Script-tagged rows: a Latin-script *name* takes AnE-LID's contextual label.

    Script alone calls every Latin token English, which is right for
    "Game of Thrones" but invents a switch for "Messi" in a Cantonese sentence.
    AnE-LID, run on a window around the name, tells the two apart. Names in the
    row's own script (中關村) stay L1, and non-entity tokens keep their script
    label. An empty label means the name was never scored; script stands.
    """
    out = []
    for (t, lang), ent, lab in zip(tagged, is_entity, ane_labels):
        if ent and lab and _LATIN.search(t):
            lang = ane_to_language(lab, first_language, second_language)
        out.append((t, lang))
    return out


def tag_tokens(tokens: list[str], first_language: str, second_language: str,
               ane_labels: list[str] | None = None,
               is_entity: list[bool] | None = None) -> list[tuple[str, str]]:
    """Tag tokens as first/second language via whichever tier the pair needs.

    ``ane_labels`` are precomputed raw AnE-LID word labels (``persist.ane_labels``
    batches them over the corpus). For ``ane`` rows they are the labels; without
    them the model runs on this row. For script-tagged rows they are optional
    and only cover the windows around Latin-script names (see
    ``_names_from_context``); ``""`` marks unscored positions.
    """
    strategy = row_strategy(tokens, first_language, second_language)
    if strategy == "ane":
        from .ane_tagger import tag_tokens_ane
        from .lexical_override import override_tagged
        tagged = _devanagari_is_hindi(
            tag_tokens_ane(tokens, first_language, second_language, labels=ane_labels),
            first_language)
        ents = is_entity if is_entity is not None and len(is_entity) == len(tagged) else None
        return override_tagged(tagged, first_language, second_language, ents)[0]
    if strategy == "unicode":
        from .unicode_tagger import tag_tokens_unicode
        tagged = tag_tokens_unicode(tokens, first_language, second_language)
        if (ane_labels is not None and is_entity is not None
                and len(ane_labels) == len(tokens) == len(is_entity)):
            tagged = _names_from_context(tagged, is_entity, ane_labels,
                                         first_language, second_language)
        return tagged
    if strategy == "bert":
        from .bert_tagger import tag_tokens_bert
        return _devanagari_is_hindi(
            tag_tokens_bert(tokens, first_language, second_language), first_language)
    if strategy == "dict":
        from .dict_tagger import tag_frequency
        return tag_frequency(tokens, first_language, second_language)
    from .model_tagger import tag_tokens_model
    return tag_tokens_model(tokens, first_language, second_language)

In [ ]:
%%writefile src/segments.py
"""Run-length encoding of tagged tokens, plus the dataset builders.

The plan leaves one decision open (line 174): whether an ``"other"`` tag
(punctuation, digits) breaks a run or is absorbed into the surrounding
language. Absorbing is cleaner for the downstream goals, so ``absorb_other``
does that - with one guard the naive version gets wrong: an ``other`` token
sitting *between two different languages* must not bridge them into a single
run. "hola, hello" is two runs, not one.

Named entities: every token keeps a *language* label, and entity-ness travels
separately as an ``is_entity`` flag. How names count is a policy, not a label
(``resolve_names``; every edge case is pinned in tests/test_name_strategies.py).
AnE-NER fragments names, so ``repair_spans`` mends them first for every policy.

* ``names="script_words"`` (default) - a name is a switch only with visible
  evidence. A Latin-script name in a non-Latin row is English ("Messi",
  "Severance Season 2" inside Arabic), since the writer chose the Latin script;
  a name in the row's own script stays L1. In Latin-script rows the name's
  words decide when they are clearly one language's ("Severance Season" ->
  English, "Union europeenne" -> French); otherwise it takes its surroundings
  ("Ayer Leo Messi marco" -> Spanish), and may not open a run on its own.
* ``names="transparent"`` - names are no language and are skipped (LinCE's
  convention). Stored as the main ablation.
* ``names="script"`` - every Latin-script name is English. Stored too; adds a
  one-word switch for each Messi / Monza / EU inside a Latin-script sentence.
* ``names="hybrid"`` - titles keep their language, other names take context.
* ``names="language"`` - names count as AnE-LID tagged them.
"""
from __future__ import annotations

import ast

import regex as re

from .routing import route
from .tokenizer import tokenize_mixed

NE = "NE"
NAME_POLICIES = ("hybrid", "language", "transparent", "script", "script_words")
# A name counts as a switch only with visible evidence: a change of script, or
# words that are clearly one language's. See tests/test_name_strategies.py.
DEFAULT_NAMES = "script_words"

# A function word: short and extremely frequent in its language (wordfreq Zipf).
# Measured: of/the/and/de/la/der/dan/ng/cua all score 6.5-7.9; the most common
# content words in candidate names (game, real, united, league) stay <= 5.8.
_FUNCTION_ZIPF = 6.4
_FUNCTION_MAXLEN = 4


def _is_function_word(word: str, language: str) -> bool:
    from .dict_tagger import LANG_TO_CODE, zipf
    code = LANG_TO_CODE.get(language)
    return (code is not None and len(word) <= _FUNCTION_MAXLEN
            and zipf(word.lower(), code) >= _FUNCTION_ZIPF)


_LATIN = re.compile(r"\p{Script=Latin}")
# Languages whose wordfreq list is written in Latin script, so a Latin-script
# name can be looked up in it. Hindi's list is Devanagari: romanized names
# score ~0 there, which would make every Bollywood name "English".
_LATIN_DICT = {"English", "Spanish", "German", "French", "Italian", "Malay",
               "Philippines", "Vietnamese"}
_MARGIN = 1.0      # Zipf gap for a word to count as clearly one language's


def _latin(tok: str) -> bool:
    return bool(_LATIN.search(tok))


def _zipf(word: str, language: str):
    if language not in _LATIN_DICT:
        return None
    from .dict_tagger import LANG_TO_CODE, zipf
    return zipf(word.lower(), LANG_TO_CODE[language])


def _word_side(word: str, first_language: str, second_language: str):
    """Which of the pair a Latin word clearly belongs to, by dictionary margin; None if unclear.

    Names sit in both frequency lists at similar levels (messi 3.8 en / 4.1 es,
    madrid 4.3 / 5.4), so only a >= 1.0 Zipf gap decides: severance, season ->
    English; europeenne, monza -> L1; messi, psg, eu -> unclear.
    """
    z1, z2 = _zipf(word, first_language), _zipf(word, second_language)
    if z1 is None or z2 is None:
        return None
    if z2 - z1 >= _MARGIN:
        return second_language
    if z1 - z2 >= _MARGIN:
        return first_language
    return None


def repair_spans(tokens, is_entity, first_language: str, second_language: str) -> list[bool]:
    """Mend AnE-NER's fragmented Latin-script names before any policy uses them.

    Measured on real output: "Severance Season 2" came back as just
    "Severance", "Game of Thrones" as "Game" + "Thrones", "The University of
    Texas" as "of Texas". Two repairs, Latin script only:

    * bridge a single function word between two name words ("Game [of] Thrones");
    * grow a name over adjacent capitalised words - and numbers to its right -
      unless the word is clearly an ordinary L1 word ("Ayer Leo Messi" keeps
      "Ayer"; German nouns like "Kommission" are never swallowed).
    Non-Latin names are left exactly as tagged.
    """
    mask = list(is_entity)
    n = len(tokens)
    for k in range(1, n - 1):
        if (not mask[k] and mask[k - 1] and mask[k + 1]
                and _latin(tokens[k - 1]) and _latin(tokens[k + 1])
                and any(_is_function_word(tokens[k], lang)
                        for lang in {first_language, second_language, "English"})):
            mask[k] = True

    def joins(tok: str, to_the_right: bool) -> bool:
        if to_the_right and tok.isdigit():
            return True
        return (_latin(tok) and tok[:1].isupper()
                and _word_side(tok, first_language, second_language) != first_language)

    changed = True
    while changed:
        changed = False
        for k in range(n):
            if mask[k]:
                continue
            after_name = k > 0 and mask[k - 1] and (_latin(tokens[k - 1]) or tokens[k - 1].isdigit())
            before_name = k + 1 < n and mask[k + 1] and _latin(tokens[k + 1])
            if (after_name and joins(tokens[k], True)) or (before_name and joins(tokens[k], False)):
                mask[k] = changed = True
    return mask


def _context_language(out, ent, i, j, own, langs):
    """An opaque name (tokens i..j-1) may not open a run of its own.

    Inside one language it takes that language. At a real switch it joins the
    side not cut off by punctuation; if neither or both are, it keeps ``own``
    when that is one of the two sides.
    """
    n = len(out)

    def neighbour(k, step):
        k += step
        cut = False
        while 0 <= k < n:
            t, lang = out[k]
            if not _is_word(t):
                cut = True
            elif not ent[k] and lang in langs:
                return lang, cut
            k += step
        return None, cut

    (prev, prev_cut), (nxt, nxt_cut) = neighbour(i, -1), neighbour(j - 1, 1)
    if prev and nxt and prev != nxt:
        if prev_cut != nxt_cut:
            return nxt if prev_cut else prev
        return own if own in (prev, nxt) else prev
    return prev or nxt or own


def _spans(ent):
    i, n = 0, len(ent)
    while i < n:
        if not ent[i]:
            i += 1
            continue
        j = i
        while j < n and ent[j]:
            j += 1
        yield i, j
        i = j


def resolve_names(tagged: list[tuple[str, str]], is_entity, first_language: str,
                  second_language: str, names: str = DEFAULT_NAMES) -> list[tuple[str, str]]:
    """Apply a names policy to language-tagged tokens (see the module docstring)."""
    if is_entity is None or names == "language":
        return list(tagged)
    if names not in NAME_POLICIES:
        raise ValueError(f"unknown names policy {names!r}; use one of {NAME_POLICIES}")
    toks = [t for t, _ in tagged]
    ent = repair_spans(toks, is_entity, first_language, second_language)
    if names == "transparent":
        return apply_entities(tagged, ent)

    from .dispatch import _hindi_is_devanagari
    from .routing import SCRIPT_DISJOINT_CONFIRMED
    script_row = (first_language in SCRIPT_DISJOINT_CONFIRMED
                  or (first_language == "Hindi" and _hindi_is_devanagari(toks)))
    langs = (first_language, second_language)
    out = list(tagged)
    for i, j in _spans(ent):
        words = [k for k in range(i, j) if _is_word(out[k][0])]
        tagged_langs = [out[k][1] for k in words if out[k][1] in langs]
        if not tagged_langs:
            continue
        own = max(langs, key=tagged_langs.count)
        latin = any(_latin(out[k][0]) for k in words)

        if names == "hybrid":
            title = len(words) >= 2 and any(_is_function_word(out[k][0], own) for k in words)
            if not title:
                own = _context_language(out, ent, i, j, own, langs)
        elif names == "script":
            if latin and second_language == "English":
                own = second_language
        elif names == "script_words" and latin:
            if script_row:
                if second_language == "English":
                    own = second_language
            else:
                votes = [_word_side(out[k][0], first_language, second_language) for k in words]
                votes = [v for v in votes if v]
                if votes and votes.count(langs[0]) != votes.count(langs[1]):
                    own = max(langs, key=votes.count)
                else:
                    own = _context_language(out, ent, i, j, own, langs)
        for k in words:
            out[k] = (out[k][0], own)
    return out

def apply_entities(tagged: list[tuple[str, str]],
                   is_entity: list[bool]) -> list[tuple[str, str]]:
    """The ``transparent`` policy: relabel entity *words* ``NE``.

    Punctuation inside a name stays ``other``.
    """
    return [(t, NE if ent and _is_word(t) else lang)
            for (t, lang), ent in zip(tagged, is_entity)]


def absorb_other(tagged: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """Reassign ``other`` tokens to the surrounding language.

    An ``other`` run takes the language of the token before it, unless that
    would merge two different languages - at a genuine switch boundary the
    punctuation joins the *following* language instead, so the boundary stays
    where it is. Leading ``other`` tokens join whatever follows. Token count is
    preserved.
    """
    out = list(tagged)
    n = len(out)
    i = 0
    while i < n:
        if out[i][1] != "other":
            i += 1
            continue
        j = i
        while j < n and out[j][1] == "other":
            j += 1
        # Look through NE tokens: they belong to neither neighbour, so the
        # boundary test must be made against the nearest real language.
        p = i - 1
        while p >= 0 and out[p][1] == NE:
            p -= 1
        q = j
        while q < n and out[q][1] in ("other", NE):
            q += 1
        prev_lang = out[p][1] if p >= 0 else None
        next_lang = out[q][1] if q < n else None
        if prev_lang is None:
            fill = next_lang
        elif next_lang is None or prev_lang == next_lang:
            fill = prev_lang
        else:
            # Genuine switch boundary: do not let punctuation bridge it.
            fill = next_lang
        if fill is not None:
            for k in range(i, j):
                out[k] = (out[k][0], fill)
        i = j
    return out


def to_segments(tagged_tokens: list[tuple[str, str]]) -> list[list]:
    """Collapse consecutive same-language tokens into ``[language, length]`` runs."""
    segments: list[list] = []
    for _, lang in tagged_tokens:
        if segments and segments[-1][0] == lang:
            segments[-1][1] += 1
        else:
            segments.append([lang, 1])
    return segments


def l2_runs(segments: list[list], second_language: str) -> list[int]:
    """Run lengths straight off the segments, punctuation included.

    Prefer :func:`l2_word_runs` for the switch-length question - segments carry
    absorbed punctuation, which inflates every run.
    """
    return [n for lang, n in segments if lang == second_language]


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def l2_word_runs(tagged: list[tuple[str, str]], second_language: str) -> list[int]:
    """Lengths of each consecutive second-language run, counting WORDS only.

    This is the original question: how many second-language tokens are used
    consecutively before switching back or ending.

    Punctuation is *transparent* - it neither breaks a run nor counts toward
    it. That matters because ``absorb_other`` hands boundary punctuation to the
    following language, so a sentence-final "." belonging to the first-language
    sentence would otherwise be counted as a second-language token and inflate
    every run by one.
    """
    runs, current = [], 0
    for token, lang in tagged:
        if not _is_word(token) or lang == NE:
            continue                      # transparent: no break, no count
        if lang == second_language:
            current += 1
        elif current:
            runs.append(current)
            current = 0
    if current:
        runs.append(current)
    return runs


def parse_utterances(cell: str) -> list[str]:
    """``data_generation_result`` is a stringified Python list of utterances."""
    try:
        value = ast.literal_eval(str(cell))
    except (ValueError, SyntaxError):
        return [str(cell)]
    if isinstance(value, list):
        return [str(v) for v in value]
    return [str(value)]


def tag_row(cell: str, first_language: str, second_language: str,
            cross_utterance: bool = True, is_entity: list[bool] | None = None,
            return_tagged: bool = False, ane_labels: list[str] | None = None,
            names: str = DEFAULT_NAMES) -> dict:
    """Tokenise, tag and run-length-encode one dataframe row.

    ``cross_utterance=True`` streams every utterance together, so a run may span
    a turn boundary; the utterance each run starts in is still recorded.
    """
    from .dispatch import row_strategy, tag_tokens

    utterances = parse_utterances(cell)

    tokens: list[str] = []
    origin: list[int] = []
    for idx, utt in enumerate(utterances):
        toks = tokenize_mixed(utt)
        tokens.extend(toks)
        origin.extend([idx] * len(toks))

    if not tokens:
        return dict(segments=[], l2_runs=[], n_switches=0, max_run=0,
                    total_l2_tokens=0, total_tokens=0, l2_token_ratio=0.0,
                    method=route(first_language, second_language),
                    unknown_rate=0.0, run_utterance_idx=[])

    strategy = row_strategy(tokens, first_language, second_language)
    tagged = tag_tokens(tokens, first_language, second_language,
                        ane_labels=ane_labels, is_entity=is_entity)
    lang_tagged = tagged                  # stored labels: always a language, policy-free
    tagged = resolve_names(tagged, is_entity, first_language, second_language, names)
    if strategy == "fasttext":
        from .model_tagger import unknown_rate
        unk = unknown_rate(tokens, first_language, second_language)
    else:
        unk = 0.0

    if not cross_utterance:
        # Break runs at turn boundaries by inserting a sentinel language.
        tagged = [
            t if i == 0 or origin[i] == origin[i - 1] else (t[0], t[1])
            for i, t in enumerate(tagged)
        ]

    tagged = absorb_other(tagged)
    segments = to_segments(tagged)
    runs = l2_word_runs(tagged, second_language)

    # Utterance index where each second-language run begins.
    run_start_idx: list[int] = []
    pos = 0
    for lang, n in segments:
        if lang == second_language:
            run_start_idx.append(origin[pos] if pos < len(origin) else -1)
        pos += n

    total_l2 = sum(runs)
    counted = sum(n for lang, n in segments if lang not in ("other", NE))
    result = dict(
        segments=segments,
        l2_runs=runs,
        n_switches=len(runs),
        max_run=max(runs) if runs else 0,
        total_l2_tokens=total_l2,
        total_tokens=len(tokens),
        l2_token_ratio=(total_l2 / counted) if counted else 0.0,
        method=strategy,
        unknown_rate=unk,
        run_utterance_idx=run_start_idx,
    )
    if return_tagged:
        result["tagged"] = lang_tagged   # pre-absorption, never NE
    return result

In [ ]:
%%writefile src/anticipation.py
"""Anticipatory labels: will the NEXT token switch language, and for how long?

The tagging pipeline answers "what language is token t?". This module turns
that into the *prediction* task: standing at token t with only tokens 1..t
visible, predict two things about t+1.

    y_sw[t]  = 1 if lang(t+1) != lang(t)          binary, defined for t < n-1
    y_dur[t] = class of the length of the run t+1 starts (Small 1-2 | Medium 3-6 | Large 7+)

Both labels are derived, never annotated - they are a deterministic function of
the token-language sequence produced by ``src.dispatch``. That means label
quality is exactly tagger quality, which is why ``quality`` travels with every
row (``exact`` for script-disjoint pairs, ``provisional`` otherwise).

Three decisions that materially change the numbers, made explicit:

1. **Punctuation is dropped, not labelled.** ``l2_word_runs`` treats
   non-alphabetic tokens as transparent - they neither break a run nor count
   toward it. The anticipatory stream uses the same rule, so a run length here
   means the same thing as a run length in the descriptive statistics. Feeding
   punctuation to the model as a predictable "switch" would inflate the
   positive class with tokens nobody wants predicted.
2. **The final token carries no label.** There is no t+1 to predict, so
   position n-1 is masked out of the loss rather than being assigned a
   convenient 0. Assigning 0 would add one free true-negative per row -
   234,172 of them - and quietly lift precision.
3. **Durations at the sequence end are right-censored.** A run still in
   progress when the text ends is *at least* its observed length. It is
   flagged ``censored``, and it is used as a duration target only when that
   lower bound already fixes the class (7+ is Large).
"""
from __future__ import annotations

import numpy as np
import regex

from .segments import DEFAULT_NAMES, absorb_other, resolve_names

# Chinese, Japanese and Korean characters. Every tokenizer we use splits at
# whitespace, so non-CJK words are the same units for all of them; CJK is not.
CJK = regex.compile(r"[\p{Han}\p{Hiragana}\p{Katakana}\p{Hangul}]")

# Language slots. The stream is binarised relative to the row's own pair, so a
# model trained across pairs sees "first vs second language", not "Spanish vs
# English" - that is what makes the universality claim testable.
L1, L2 = 0, 1

# A piece or token that carries no language: punctuation, digits, a bare
# SentencePiece "▁". It stays in the input but is transparent to the labels -
# it neither breaks a run nor counts toward one. Equal to streaming.PAD_LANG.
NONE = 2

# Duration classes from the project requirement: Small 1-2, Medium 3-6,
# Large 7+ tokens. The integer length is kept as y_len; the class is derived.
DUR_EDGES = (2, 6)                     # inclusive upper bounds of small, medium
DUR_LABELS = ["small", "medium", "large"]
N_DUR = len(DUR_LABELS)
LARGE_MIN = DUR_EDGES[-1] + 1          # a cut-off run this long is certainly large
# Two-class view (--dur-classes 2): Short = Small + Medium (1-6) vs Large (7+).
# The scored positions are the same: a cut-off run is certain only once it is 7+.
BINARY_LABELS = ["short", "large"]


def dur_labels(n_classes: int) -> list[str]:
    """Class names for a duration head with ``n_classes`` outputs (3 or 2)."""
    return {len(DUR_LABELS): DUR_LABELS, 2: BINARY_LABELS}[n_classes]


def collapse_duration(y_dur):
    """3-class ``y_dur`` -> Short (0) / Large (1); -1 (no label) stays -1.

    Works on a numpy array or a torch tensor and returns the same type.
    """
    large = y_dur == N_DUR - 1
    out = (y_dur >= 0) * 1 - 1 + large * 1      # -1 -> -1, small/medium -> 0, large -> 1
    return out.astype(y_dur.dtype) if hasattr(out, "astype") else out.to(y_dur.dtype)

def is_word(token: str) -> bool:
    """A token counts toward a run only if it contains a letter."""
    return any(ch.isalpha() for ch in token)


# How named entities count - see segments.py. Scripts set this from --names.
NAMES = DEFAULT_NAMES


def token_slots(tokens, token_lang, first_language: str, second_language: str,
                is_entity=None, names: str | None = None) -> list[int]:
    """Language slot (L1 / L2 / NONE) for every stored token of a row.

    Applies the names policy and ``absorb_other`` exactly as the word stream
    always has; a token with no letters, or no resolvable language, is NONE.
    """
    tagged = resolve_names(list(zip(tokens, token_lang)), is_entity,
                           first_language, second_language, names or NAMES)
    out = []
    for tok, lang in absorb_other(tagged):
        if not is_word(tok):
            out.append(NONE)
        elif lang == second_language:
            out.append(L2)
        elif lang == first_language:
            out.append(L1)
        else:
            out.append(NONE)                # NE, or unresolved: drop rather than guess
    return out


def word_stream(tokens, token_lang, first_language: str, second_language: str,
                is_entity=None, names: str | None = None):
    """Collapse a tagged row into the (words, language-slot) stream to model.

    Keeps only tokens with a language slot, so punctuation and digits are
    dropped from the stream entirely.
    """
    slots = token_slots(tokens, token_lang, first_language, second_language,
                        is_entity, names)
    keep = [i for i, s in enumerate(slots) if s != NONE]
    return ([tokens[i] for i in keep],
            np.asarray([slots[i] for i in keep], dtype=np.int8))

def run_starts_and_lengths(langs: np.ndarray):
    """Index where each run begins, and its length. Length n == 0 -> empty."""
    n = len(langs)
    if n == 0:
        return np.empty(0, np.int64), np.empty(0, np.int64)
    boundary = np.flatnonzero(langs[1:] != langs[:-1]) + 1
    starts = np.concatenate([[0], boundary])
    ends = np.concatenate([boundary, [n]])
    return starts, ends - starts


def dur_class(length: int) -> int:
    """Map an integer run length to its DUR_LABELS index."""
    for i, upper in enumerate(DUR_EDGES):
        if length <= upper:
            return i
    return N_DUR - 1


def anticipatory_labels(langs: np.ndarray) -> dict:
    """Per-position labels for a language-slot sequence; NONE slots are transparent.

    Position ``t`` describes the next language-bearing position after it.

    * ``y_sw``      0/1 switch indicator (-1 where undefined: NONE slots, last one)
    * ``y_len``     integer length of the run the switch starts (0 elsewhere)
    * ``y_dur``     ``dur_class(y_len)`` (-1 elsewhere)
    * ``censored``  that run is still open at the end of the sequence
    * ``dur_known`` a duration target exists and its class is certain
    """
    n = len(langs)
    y_sw = np.full(n, -1, dtype=np.int8)
    y_len = np.zeros(n, dtype=np.int32)
    y_dur = np.full(n, -1, dtype=np.int8)
    censored = np.zeros(n, dtype=bool)
    dur_known = np.zeros(n, dtype=bool)
    out = dict(y_sw=y_sw, y_len=y_len, y_dur=y_dur, censored=censored,
               dur_known=dur_known)
    idx = np.flatnonzero(np.asarray(langs) != NONE)
    if len(idx) < 2:
        return out
    lb = np.asarray(langs)[idx]
    y_sw[idx[:-1]] = (lb[1:] != lb[:-1]).astype(np.int8)
    starts, lengths = run_starts_and_lengths(lb)
    m = len(lb)
    # A run starting at language-bearing index s>0 is anticipated at idx[s-1].
    for s, ln in zip(starts[1:], lengths[1:]):
        t = idx[s - 1]
        cut = bool(s + ln == m)
        y_len[t] = ln
        y_dur[t] = dur_class(int(ln))
        censored[t] = cut
        dur_known[t] = (not cut) or ln >= LARGE_MIN
    return out

def row_example(row) -> dict | None:
    """Build one word-level training example from a ``load_tagged``-shaped row."""
    words, langs = word_stream(
        row["tokens"], row["token_lang"],
        row["first_language"], row["second_language"],
        is_entity=row.get("is_entity"),
    )
    if len(words) < 2:
        return None
    return dict(
        row_id=int(row["row_id"]),
        words=words,
        langs=langs,
        shared=np.array([not CJK.search(w) for w in words], dtype=bool),
        first_language=row["first_language"],
        second_language=row["second_language"],
        quality=row.get("quality", "unknown"),
        **anticipatory_labels(langs),
    )


def _word_features(words, langs):
    """``unit_end`` (last piece of each word) and ``lang_in`` (the language of the
    last *finished* word) for every piece.

    ``lang_in`` at piece t only uses words that ended before t's own word
    began, so it is a legal causal input: a word's language is known once all
    of its pieces have been read. NONE (the pad slot) where no word has finished.
    """
    n = len(langs)
    unit_end = np.zeros(n, dtype=bool)
    lang_in = np.full(n, NONE, dtype=np.int8)
    wlang, cur, done = {}, None, NONE
    for i in range(n):
        w = int(words[i])
        if w >= 0 and w != cur:
            if cur is not None and cur in wlang:
                done = wlang[cur]
            cur = w
        if w >= 0 and langs[i] != NONE:
            wlang.setdefault(w, langs[i])
        lang_in[i] = done
    nxt = None
    for i in range(n - 1, -1, -1):
        if words[i] >= 0:
            unit_end[i] = words[i] != nxt
            nxt = words[i]
    return unit_end, lang_in


def gap_landing(langs: np.ndarray, blank: np.ndarray) -> dict[int, int]:
    """Where each label moves under ``decide_at="gap"``: ``{source: landing}``.

    A label normally sits on the last language-bearing piece before the next
    one (index ``a``), so in ``home , 然后`` the model decides at ``home``
    without having read the comma. With the gap rule it moves to the last
    piece of the gap that is not blank (a bare space marker). The model then
    decides after reading the punctuation.

    The bare ``▁`` is skipped because XLM-R emits it only when the next
    character does not merge with the marker (CJK, for example), so landing on
    it would leak the next word's script. Adjacent pairs, and gaps holding only
    blanks, keep their label where it is and are left out of the map.
    """
    langs = np.asarray(langs)
    blank = np.asarray(blank, dtype=bool)
    idx = np.flatnonzero(langs != NONE)
    out = {}
    for a, b in zip(idx[:-1], idx[1:]):
        for j in range(b - 1, a, -1):
            if not blank[j]:
                out[int(a)] = j
                break
    return out


# Value a position holds when it carries no label, per moved key.
_UNLABELLED = dict(y_sw=-1, y_len=0, y_dur=-1, censored=False, dur_known=False,
                   shared=False, unit_end=False)


def _relocate(ex: dict, moves: dict[int, int]) -> None:
    """Move every per-position label from source to landing, in place."""
    if not moves:
        return
    src = np.fromiter(moves.keys(), dtype=np.int64)
    dst = np.fromiter(moves.values(), dtype=np.int64)
    for k, empty in _UNLABELLED.items():
        arr = ex.get(k)
        if arr is None:
            continue
        vals = arr[src].copy()
        arr[src] = empty
        arr[dst] = vals
    ex["cur_lang"][dst] = ex["langs"][src]


def subword_example(row, turn_sep: bool = False, decide_at: str = "word",
                    blank_ids=None) -> dict | None:
    """Build one sub-word training example from a ``data/subword.parquet`` row.

    ``turn_sep`` inserts a separator piece (id -1, language NONE, so no label
    moves) before the first piece of every utterance after the first.

    ``decide_at="gap"`` moves each label from the last language-bearing piece
    onto the punctuation that follows it (``gap_landing``); ``blank_ids`` are
    the tokenizer's bare space-marker ids. Positives, durations and the
    number of scored positions are unchanged. ``cur_lang`` keeps the language
    of the word the label belongs to, for the per-direction report.
    """
    if decide_at not in ("word", "gap"):
        raise ValueError(f"decide_at is 'word' or 'gap', not {decide_at!r}")
    if decide_at == "gap" and blank_ids is None:
        raise ValueError("decide_at='gap' needs blank_ids (the tokenizer's bare space-marker ids)")
    ids = np.asarray(row["piece_ids"], dtype=np.int64)
    langs = np.asarray(row["piece_lang"], dtype=np.int8)
    n = len(ids)
    shared = (np.asarray(row["word_end"], dtype=bool) if row.get("word_end") is not None
              else np.zeros(n, dtype=bool))
    words = np.asarray(row["piece_word"], dtype=np.int64) if row.get("piece_word") is not None else None
    utts = np.asarray(row["piece_utt"], dtype=np.int64) if row.get("piece_utt") is not None else None
    if turn_sep:
        if utts is None:
            raise ValueError("--turn-sep needs piece_utt: rebuild the labels with build_subword.py")
        at = np.flatnonzero(np.r_[False, utts[1:] != utts[:-1]]) if n else np.empty(0, int)
        ids, langs = np.insert(ids, at, -1), np.insert(langs, at, NONE)
        shared = np.insert(shared, at, False)
        if words is not None:
            words = np.insert(words, at, -1)
    if int((langs != NONE).sum()) < 2:
        return None
    unit_end, lang_in = _word_features(words, langs) if words is not None else (None, None)
    ex = dict(
        row_id=int(row["row_id"]),
        ids=ids,
        langs=langs,
        shared=shared,
        unit_end=unit_end,
        lang_in=lang_in,
        first_language=row["first_language"],
        second_language=row["second_language"],
        quality=row.get("quality", "unknown"),
        **anticipatory_labels(langs),
    )
    if decide_at == "gap":
        ex["cur_lang"] = langs.copy()
        _relocate(ex, gap_landing(langs, np.isin(ids, np.asarray(blank_ids, dtype=np.int64))))
    return ex

In [ ]:
%%writefile src/subword.py
"""Sub-word pieces for the anticipation task.

Each XLM-R piece takes the language of the word it sits in. The word labels
are the final per-token labels (``token_lang`` after the names policy), and
the original text comes from the SwitchLingua CSVs, joined on ``row_id``. The
stored tokens were produced by ``tokenize_mixed`` from that text, so an
in-order search recovers each token's character span exactly.

A piece's language is the majority slot of the word characters it covers
(ties go to L1; a piece covering two languages is counted, not an error). A
piece covering no word characters - punctuation, digits, spaces, or the bare
SentencePiece "▁" that XLM-R emits with the next character's offsets - is
NONE and transparent to the labels. Merged CJK pieces such as 東京 need no
special case: their characters share a language.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .anticipation import CJK, L1, L2, NONE, anticipatory_labels, token_slots
from .tokenizer import tokenize_mixed

XLMR_VOCAB = 250_002
# Space markers: SentencePiece (XLM-R) "▁", byte-level BPE (RoBERTa) "Ġ" and "Ċ".
# WordPiece (mBERT) has none; its "##" continuations are ordinary pieces.
MARKERS = ("▁", "Ġ", "Ċ")


def is_blank(piece: str) -> bool:
    """A piece that is only a space marker carries no characters of its own."""
    for m in MARKERS:
        piece = piece.replace(m, "")
    return piece == ""


def blank_piece_ids(tokenizer_name: str = "xlm-roberta-base") -> list[int]:
    """Ids of every bare space-marker piece in a tokenizer's vocabulary
    (``▁`` = 6 in XLM-R), for ``anticipation.gap_landing``."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(tokenizer_name)
    return sorted(i for t, i in tok.get_vocab().items() if is_blank(t))
LABEL_KEYS = ("y_sw", "y_len", "y_dur", "censored", "dur_known")


def align_words(utterances: list[str], tokens: list[str]):
    """``(utterance, start, end)`` for every stored token, in order, or None."""
    spans, k = [], 0
    for u, text in enumerate(utterances):
        pos = 0
        for t in tokenize_mixed(text):
            if k >= len(tokens) or tokens[k] != t:
                return None
            j = text.find(t, pos)
            if j < 0:
                return None
            spans.append((u, j, j + len(t)))
            pos = j + len(t)
            k += 1
    return spans if k == len(tokens) else None


def piece_languages(offsets, blank, char_slots: np.ndarray):
    """Slot per piece from the word slots of the characters it covers."""
    langs, mixed = [], 0
    for (s, e), is_blank in zip(offsets, blank):
        if is_blank:
            langs.append(NONE)
            continue
        seg = char_slots[s:e]
        n1, n2 = int((seg == L1).sum()), int((seg == L2).sum())
        if n1 == 0 and n2 == 0:
            langs.append(NONE)
            continue
        mixed += int(n1 > 0 and n2 > 0)
        langs.append(L2 if n2 > n1 else L1)
    return langs, mixed


def subword_row(utterances, tokens, token_lang, first_language: str,
                second_language: str, tokenizer, is_entity=None):
    """Piece ids and per-piece language slots for one row, or None if unaligned."""
    tokens, token_lang = list(tokens), list(token_lang)
    spans = align_words(utterances, tokens)
    if spans is None:
        return None
    slots = token_slots(tokens, token_lang, first_language, second_language,
                        None if is_entity is None else list(is_entity))
    char_slots = [np.full(len(u), -1, dtype=np.int8) for u in utterances]
    char_tok = [np.full(len(u), -1, dtype=np.int64) for u in utterances]
    for k, ((u, s, e), slot) in enumerate(zip(spans, slots)):
        char_tok[u][s:e] = k
        if slot != NONE:
            char_slots[u][s:e] = slot
    # Word ends: the last piece of each non-CJK word. Every tokenizer we use
    # splits at whitespace, so these are the same words for all of them - the
    # positions on which models with different tokenizers can be compared.
    shared = {k for k, (t, s) in enumerate(zip(tokens, slots)) if s != NONE and not CJK.search(t)}
    ids, pieces, langs, word_end, piece_word, piece_utt, mixed = [], [], [], [], [], [], 0
    for u, text in enumerate(utterances):
        if not text:
            continue
        enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
        strs = tokenizer.convert_ids_to_tokens(enc["input_ids"])
        blank = [is_blank(p) for p in strs]
        lg, m = piece_languages(enc["offset_mapping"], blank, char_slots[u])
        last = {}
        for i, ((s, e), b) in enumerate(zip(enc["offset_mapping"], blank)):
            covered = set(char_tok[u][s:e].tolist()) - {-1} if not b else set()
            piece_word.append(max(covered) if covered else -1)    # the word it belongs to
            piece_utt.append(u)
            for k in covered & shared:
                last[k] = len(ids) + i
        ends = set(last.values())
        word_end.extend(len(ids) + i in ends for i in range(len(strs)))
        ids.extend(enc["input_ids"])
        pieces.extend(strs)
        langs.extend(lg)
        mixed += m
    return dict(piece_ids=ids, pieces=pieces, piece_lang=langs, word_end=word_end,
                piece_word=piece_word, piece_utt=piece_utt, mixed_pieces=mixed)


def build_frame(frame: pd.DataFrame, texts: dict, tokenizer, max_fail: float = 0.001):
    """Sub-word rows for every tagged row; misaligned rows are skipped and counted."""
    out, failed, mixed = [], 0, 0
    has_ent = "is_entity" in frame.columns
    for r in frame.itertuples(index=False):
        res = subword_row(texts[int(r.row_id)], r.tokens, r.token_lang,
                          r.first_language, r.second_language, tokenizer,
                          is_entity=r.is_entity if has_ent else None)
        if res is None:
            failed += 1
            continue
        mixed += res["mixed_pieces"]
        lab = anticipatory_labels(np.asarray(res["piece_lang"], dtype=np.int8))
        out.append(dict(row_id=int(r.row_id), first_language=r.first_language,
                        second_language=r.second_language, quality=r.quality, **res,
                        **{k: lab[k].tolist() for k in LABEL_KEYS}))
    rate = failed / max(len(frame), 1)
    if rate > max_fail:
        raise RuntimeError(f"{failed:,} of {len(frame):,} rows failed to align "
                           f"({rate:.2%} > {max_fail:.2%})")
    stats = dict(rows=len(out), failed=failed, mixed_pieces=mixed,
                 pieces=sum(len(x["piece_ids"]) for x in out))
    return pd.DataFrame(out), stats


def write_parquet(frame: pd.DataFrame, path: str, row_group_size: int = 8192) -> None:
    import pyarrow as pa
    import pyarrow.parquet as pq
    schema = pa.schema([
        ("row_id", pa.int64()), ("first_language", pa.string()),
        ("second_language", pa.string()), ("quality", pa.string()),
        ("pieces", pa.list_(pa.string())), ("piece_ids", pa.list_(pa.int32())),
        ("piece_lang", pa.list_(pa.int8())),
        ("y_sw", pa.list_(pa.int8())), ("y_len", pa.list_(pa.int32())),
        ("y_dur", pa.list_(pa.int8())), ("censored", pa.list_(pa.bool_())),
        ("dur_known", pa.list_(pa.bool_())), ("word_end", pa.list_(pa.bool_())),
        ("piece_word", pa.list_(pa.int32())), ("piece_utt", pa.list_(pa.int16())),
        ("mixed_pieces", pa.int32()),
    ])
    table = pa.Table.from_pandas(frame[schema.names], schema=schema, preserve_index=False)
    pq.write_table(table, path, compression="snappy", row_group_size=row_group_size)


def prefix_consistency(text: str, tokenizer):
    """Does tokenizing the prefix alone reproduce each piece of the full text?

    The stored pieces come from tokenizing the whole utterance, and SentencePiece
    may segment the characters before t differently once it has seen what comes
    after them. A model that really streams can only tokenize the prefix, so a
    piece that changes is (weak) information from the future.

    Returns ``(ids, last_ok, all_ok)``, one flag per piece: ``last_ok`` if the
    prefix ``text[:end]`` ends in the same piece (what a current-token model
    reads), ``all_ok`` if it reproduces every piece so far (what a full-prefix
    model reads). Blank pieces carry the next character's offsets and are never
    scored; they are reported consistent.
    """
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc["input_ids"])
    blank = [is_blank(p) for p in tokenizer.convert_ids_to_tokens(ids)]
    prefixes = [text[:e] for _, e in enc["offset_mapping"]]
    redo = tokenizer(prefixes, add_special_tokens=False)["input_ids"] if prefixes else []
    last_ok, all_ok = [], []
    for i, (p, b) in enumerate(zip(redo, blank)):
        if b:
            last_ok.append(True)
            all_ok.append(True)
            continue
        last_ok.append(p[-1:] == ids[i:i + 1])
        all_ok.append(p == ids[:i + 1])
    return ids, last_ok, all_ok

In [ ]:
%%writefile src/token_labels.py
"""Readable per-token labels: the form shared with the team.

One row per conversation, with parallel lists aligned to its units (words, or
XLM-R pieces):

    lang       "L1" | "L2" | "-"   ("-" = punctuation/digits, skipped by the labels)
    y_sw       1 if the NEXT unit switches language, 0 if not, -1 where undefined
    y_len      at a switch, how many units the new language lasts (0 elsewhere)
    y_dur      class of y_len: "small" 1-2 | "medium" 3-6 | "large" 7+ ("" elsewhere)
    censored   the new run is still open when the text ends (y_len is a lower bound)
    dur_known  the class is certain: the run closed, or it is cut off but already 7+

Everything is derived from the final word labels by ``anticipation`` - these
functions only change the representation.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .anticipation import DUR_LABELS, anticipatory_labels, token_slots

LANG_CODES = np.array(["L1", "L2", "-"])
META = ["row_id", "first_language", "second_language", "split", "quality"]


def split_of(row_id: int) -> str:
    """The row-level split used everywhere: row_id % 10 -> 0-7 train, 8 val, 9 test."""
    k = int(row_id) % 10
    return "train" if k < 8 else "val" if k == 8 else "test"


def readable(units, slots) -> dict:
    """Labels for one row as plain lists, aligned to ``units``."""
    slots = np.asarray(slots, dtype=np.int8)
    lab = anticipatory_labels(slots)
    return {
        "lang": LANG_CODES[slots].tolist(),
        "y_sw": lab["y_sw"].astype(int).tolist(),
        "y_len": lab["y_len"].astype(int).tolist(),
        "y_dur": [DUR_LABELS[d] if d >= 0 else "" for d in lab["y_dur"]],
        "censored": lab["censored"].tolist(),
        "dur_known": lab["dur_known"].tolist(),
    }


def _meta(r) -> dict:
    return {"row_id": int(r.row_id), "first_language": r.first_language,
            "second_language": r.second_language, "split": split_of(r.row_id),
            "quality": r.quality}


def word_labels(frame: pd.DataFrame, names: str | None = None) -> pd.DataFrame:
    """Per-word labels from a tagged frame (``data/tagged.parquet`` columns)."""
    rows = []
    for r in frame.itertuples(index=False):
        ent = list(r.is_entity)
        slots = token_slots(list(r.tokens), list(r.token_lang), r.first_language,
                            r.second_language, ent, names)
        rows.append({**_meta(r), "tokens": list(r.tokens), **readable(r.tokens, slots),
                     "is_entity": [int(e) for e in ent]})
    return pd.DataFrame(rows)


def subword_labels(frame: pd.DataFrame) -> pd.DataFrame:
    """Per-piece labels from a ``data/subword.parquet``-shaped frame."""
    rows = []
    for r in frame.itertuples(index=False):
        rows.append({**_meta(r), "pieces": list(r.pieces),
                     **readable(r.pieces, r.piece_lang), "piece_ids": list(r.piece_ids)})
    return pd.DataFrame(rows)


def to_subword_frame(labels: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    """The training format (``data/subword.parquet`` columns) from ``token_labels_subword``.

    Rows are shuffled with ``seed`` so streaming is not a curriculum by language;
    ``row_id`` (and so the train/val/test split) travels with each row.
    """
    slot = {"L1": 0, "L2": 1, "-": 2}
    out = pd.DataFrame({
        "row_id": labels.row_id.astype(int).to_numpy(),
        "first_language": labels.first_language.to_numpy(),
        "second_language": labels.second_language.to_numpy(),
        "quality": labels.quality.to_numpy(),
        "piece_ids": [np.asarray(x, dtype=np.int32) for x in labels.piece_ids],
        "piece_lang": [np.asarray([slot[s] for s in x], dtype=np.int8) for x in labels.lang],
    })
    return out.iloc[np.random.default_rng(seed).permutation(len(out))].reset_index(drop=True)

In [ ]:
%%writefile src/metrics.py
"""Anticipatory metrics.

``switch`` is the rare class (~5% of positions), so accuracy is useless here -
a model that never predicts a switch scores 95%. Everything below is reported
on the positive class.

**Anticipatory F1** is the harmonic mean of precision and recall for
"token t+1 begins a new language", evaluated only at positions with a defined
label and a full causal prefix.
"""
from __future__ import annotations

import numpy as np


def prf(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Precision / recall / F1 on the positive class, plus the raw counts."""
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int(np.sum(y_true & y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    n = tp + fp + fn + tn
    return dict(precision=p, recall=r, f1=f1, tp=tp, fp=fp, fn=fn, tn=tn,
                n=n, base_rate=(tp + fn) / n if n else 0.0,
                accuracy=(tp + tn) / n if n else 0.0)


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson 95% interval - used for the small per-language slices."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))


def by_group(y_true, y_pred, groups) -> dict[str, dict]:
    """Per-group PRF, for slicing by language pair or switch direction."""
    groups = np.asarray(groups)
    return {str(g): prf(np.asarray(y_true)[groups == g],
                        np.asarray(y_pred)[groups == g])
            for g in np.unique(groups)}


def macro_f1(per_group: dict[str, dict]) -> float:
    """Unweighted mean F1 - the universality number.

    Cantonese is 51% of the corpus, so a micro average measures Cantonese.
    A macro average over language pairs is what says whether the approach
    transfers.
    """
    return float(np.mean([g["f1"] for g in per_group.values()])) if per_group else 0.0


def duration_scores(y_true, y_pred, n_classes: int = 3) -> dict:
    """Duration-class accuracy, macro-F1 over classes, confusion (rows = truth)."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    conf = np.zeros((n_classes, n_classes), dtype=np.int64)
    if len(y_true) == 0:
        return dict(n=0, accuracy=0.0, within_one=0.0, macro_f1=0.0,
                    per_class_f1=[0.0] * n_classes, confusion=conf.tolist())
    np.add.at(conf, (y_true, y_pred), 1)
    f1 = []
    for c in range(n_classes):
        tp = conf[c, c]
        fp, fn = conf[:, c].sum() - tp, conf[c, :].sum() - tp
        f1.append(0.0 if tp == 0 else float(2 * tp / (2 * tp + fp + fn)))
    return dict(
        n=int(len(y_true)),
        accuracy=float(np.trace(conf) / len(y_true)),
        within_one=float(np.mean(np.abs(y_true - y_pred) <= 1)),
        macro_f1=float(np.mean(f1)),
        per_class_f1=f1,
        confusion=conf.tolist(),
    )


def _macro_f1_fast(y_true, y_pred, n_classes: int) -> float:
    conf = np.bincount(y_true * n_classes + y_pred, minlength=n_classes * n_classes)
    conf = conf.reshape(n_classes, n_classes)
    tp = np.diag(conf)
    denom = conf.sum(0) + conf.sum(1)
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))


def apply_class_offsets(prob, offsets) -> np.ndarray:
    """``argmax(log p + offsets)``: the duration decision with per-class biases."""
    return (np.log(np.clip(np.asarray(prob, dtype=float), 1e-12, None))
            + np.asarray(offsets)).argmax(-1)


def tune_class_offsets(prob, y_true, span: float = 2.0, step: float = 0.1) -> tuple:
    """Per-class log-offsets that maximise macro-F1, fit on a held-out split.

    The argmax of a model trained on imbalanced classes leans toward the
    majority; adding a bias per class moves the decision boundaries without
    retraining. The first class is fixed at 0 (only differences matter). A grid
    over the rest is refined once at a tenth of the step. Fit on VAL, apply to
    test unchanged. Returns ``(offsets, val macro-F1)``.
    """
    logp = np.log(np.clip(np.asarray(prob, dtype=float), 1e-12, None))
    y = np.asarray(y_true, dtype=np.int64)
    k = logp.shape[1]
    if len(y) == 0:
        return np.zeros(k), 0.0
    best = (_macro_f1_fast(y, logp.argmax(1), k), np.zeros(k))
    for centre, half, st in ((np.zeros(k), span, step), (None, step, step / 10)):
        c = best[1] if centre is None else centre
        axes = [np.arange(c[i] - half, c[i] + half + st / 2, st) for i in range(1, k)]
        for combo in np.array(np.meshgrid(*axes, indexing="ij")).reshape(k - 1, -1).T:
            off = np.r_[0.0, combo]
            f = _macro_f1_fast(y, (logp + off).argmax(1), k)
            if f > best[0] + 1e-12:
                best = (f, off)
    return best[1], best[0]


def end_to_end_duration(y_true, y_pred, fired, n_classes: int = 3) -> dict:
    """Duration at true switch points, counting a missed switch as a wrong class.

    ``fired`` says whether the switch head predicted the switch at that point.
    Accuracy is the fraction where the switch was predicted *and* the class is
    right; per-class F1 treats a missed switch as a false negative for its true
    class and a false positive for none. False alarms at non-switch positions
    are not counted here; switch precision already measures them.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    fired = np.asarray(fired, dtype=bool)
    if len(y_true) == 0:
        return dict(n=0, accuracy=0.0, macro_f1=0.0, per_class_f1=[0.0] * n_classes,
                    switch_recall=0.0)
    hit = fired & (y_pred == y_true)
    f1 = []
    for c in range(n_classes):
        tp = int((hit & (y_true == c)).sum())
        fp = int((fired & (y_pred == c) & (y_true != c)).sum())
        fn = int(((y_true == c) & ~hit).sum())
        f1.append(0.0 if tp == 0 else float(2 * tp / (2 * tp + fp + fn)))
    return dict(n=int(len(y_true)), accuracy=float(hit.mean()), macro_f1=float(np.mean(f1)),
                per_class_f1=f1, switch_recall=float(fired.mean()))


def confusion_table(conf, labels) -> str:
    rows = [dict(true=lab, **{f"pred_{p}": f"{c:,}" for p, c in zip(labels, row)})
            for lab, row in zip(labels, conf)]
    return fmt_table(rows, ["true"] + [f"pred_{p}" for p in labels],
                     "Duration confusion (rows = truth)")


def fmt_table(rows: list[dict], cols: list[str], title: str = "") -> str:
    """Plain-text table - these numbers go straight into the write-up."""
    head = [c for c in cols]
    widths = [max(len(h), *(len(f"{r.get(h, '')}") for r in rows)) for h in head]
    line = "  ".join(h.ljust(w) for h, w in zip(head, widths))
    out = ([title] if title else []) + [line, "  ".join("-" * w for w in widths)]
    for r in rows:
        out.append("  ".join(f"{r.get(h, '')}".ljust(w) for h, w in zip(head, widths)))
    return "\n".join(out)

In [ ]:
%%writefile src/evaluation.py
"""The one evaluation report every model is scored with.

Switch prediction is scored on the positive class at every labelled position:
precision, recall and F1 with Wilson 95% intervals, overall, per language
pair, macro-averaged across pairs, and per direction. Direction is where the
switch goes: a position whose current token is L1 predicts a switch "into
L2". Duration is scored at true switch points whose class is certain, with
3-class accuracy, macro-F1 and a confusion matrix.

Pairs are keyed "L1->L2", so German->French and German->English are never
merged.
"""
from __future__ import annotations

import numpy as np

from .anticipation import N_DUR, dur_labels
from .metrics import duration_scores, end_to_end_duration, fmt_table, prf, wilson

PAIR_KEYS = [
    "Arabic->English", "Cantonese->English", "Chinese->English", "French->English",
    "German->English", "German->French", "Hindi->English", "Italian->English",
    "Japanese->English", "Korean->English", "Malay->English", "Philippines->English",
    "Russian->English", "Spanish->English", "Vietnamese->English",
]
DIRECTIONS = {0: "into L2", 1: "into L1"}


def pair_key(first_language: str, second_language: str) -> str:
    return f"{first_language}->{second_language}"


def switch_block(truth, pred) -> dict:
    s = prf(truth, pred)
    s["precision_ci"] = wilson(s["tp"], s["tp"] + s["fp"])
    s["recall_ci"] = wilson(s["tp"], s["tp"] + s["fn"])
    return s


def duration_by_pair(dur_true, dur_pred, dur_pairs, fired=None, n_dur: int = N_DUR) -> dict:
    """``duration_scores`` on each pair's switch points, plus the end-to-end
    score when ``fired`` (did the switch head fire there) is given."""
    dt, dp, dpair = np.asarray(dur_true), np.asarray(dur_pred), np.asarray(dur_pairs)
    out = {}
    for p in np.unique(dpair):
        m = dpair == p
        block = duration_scores(dt[m], dp[m], n_dur)
        if fired is not None:
            block["end_to_end"] = end_to_end_duration(dt[m], dp[m], np.asarray(fired)[m], n_dur)
        out[PAIR_KEYS[int(p)]] = block
    return out


def full_report(truth, pred, pairs, langs, dur_true=None, dur_pred=None,
                dur_pairs=None, dur_langs=None, dur_fired=None, n_dur: int = N_DUR) -> dict:
    """Everything reported for one model on one split.

    ``pairs`` are indices into ``PAIR_KEYS``; ``langs`` is the current token's
    slot (0 = L1, 1 = L2) at each scored position. The ``dur_*`` arrays are
    aligned with ``dur_true`` (true switch points whose class is certain):
    ``dur_pairs`` and ``dur_langs`` split duration by pair and direction, and
    ``dur_fired`` (did the switch head fire there) adds the end-to-end score. ``n_dur`` is the number of duration
    classes: 3, or 2 for Short / Large (``--dur-classes 2``).
    """
    truth, pred = np.asarray(truth).astype(int), np.asarray(pred).astype(int)
    pairs, langs = np.asarray(pairs), np.asarray(langs)
    per_pair = {PAIR_KEYS[int(p)]: switch_block(truth[pairs == p], pred[pairs == p])
                for p in np.unique(pairs)}
    out = dict(
        overall=switch_block(truth, pred),
        per_pair=per_pair,
        macro_f1=float(np.mean([v["f1"] for v in per_pair.values()])) if per_pair else 0.0,
        per_direction={name: switch_block(truth[langs == k], pred[langs == k])
                       for k, name in DIRECTIONS.items() if (langs == k).any()},
    )
    if dur_true is not None:
        dt, dp = np.asarray(dur_true), np.asarray(dur_pred)
        out["duration"] = duration_scores(dt, dp, n_dur)
        fired = None if dur_fired is None else np.asarray(dur_fired, dtype=bool)
        if fired is not None:
            out["duration_end_to_end"] = end_to_end_duration(dt, dp, fired, n_dur)
        if dur_pairs is not None:
            out["duration_per_pair"] = duration_by_pair(dt, dp, dur_pairs, fired, n_dur)
        if dur_langs is not None:
            dlang = np.asarray(dur_langs)
            out["duration_per_direction"] = {
                name: duration_scores(dt[dlang == k], dp[dlang == k], n_dur)
                for k, name in DIRECTIONS.items() if (dlang == k).any()}
    return out


def _row(name, s) -> dict:
    return dict(name=name, n=f"{s['n']:,}", base=f"{s['base_rate']:.3f}",
                P=f"{s['precision']:.3f}", R=f"{s['recall']:.3f}", F1=f"{s['f1']:.3f}")


def format_report(report: dict, title: str) -> str:
    cols = ["name", "n", "base", "P", "R", "F1"]
    o = report["overall"]
    parts = [f"{title}\n  switch F1 {o['f1']:.3f}  (P {o['precision']:.3f}, R {o['recall']:.3f}, "
             f"recall 95% CI [{o['recall_ci'][0]:.3f}, {o['recall_ci'][1]:.3f}])  "
             f"| macro-F1 over pairs {report['macro_f1']:.3f}"]
    parts.append(fmt_table([_row(k, v) for k, v in report["per_direction"].items()], cols,
                           "\nBy direction:"))
    pp = sorted(report["per_pair"].items(), key=lambda kv: -kv[1]["f1"])
    parts.append(fmt_table([_row(k, v) for k, v in pp], cols, "\nBy language pair:"))
    parts.extend(duration_parts(report))
    return "\n".join(parts)


def duration_parts(report: dict) -> list[str]:
    """The duration lines of ``format_report``: pooled, end to end, per pair, per direction."""
    parts = []
    if "duration" in report:
        d = report["duration"]
        labels = dur_labels(len(d["per_class_f1"]))
        parts.append(f"\nDuration (true switch points, n={d['n']:,}): {len(labels)}-class accuracy "
                     f"{d['accuracy']:.3f} | macro-F1 {d['macro_f1']:.3f} | within one "
                     f"{d['within_one']:.3f} | per-class F1 "
                     + ", ".join(f"{c} {f:.3f}" for c, f in zip(labels, d["per_class_f1"])))
        if "duration_end_to_end" in report:
            e = report["duration_end_to_end"]
            parts.append(f"End to end (switch predicted AND class right): accuracy "
                         f"{e['accuracy']:.3f} | macro-F1 {e['macro_f1']:.3f} | switch "
                         f"recall at these points {e['switch_recall']:.3f}")
    if "duration_per_pair" in report:
        first = next(iter(report["duration_per_pair"].values()), {"per_class_f1": [0] * N_DUR})
        labels = dur_labels(len(first["per_class_f1"]))
        rows = []
        for k, d in sorted(report["duration_per_pair"].items(), key=lambda kv: -kv[1]["macro_f1"]):
            row = dict(name=k, n=f"{d['n']:,}", acc=f"{d['accuracy']:.3f}",
                       macroF1=f"{d['macro_f1']:.3f}",
                       **{f"F1_{c}": f"{f:.3f}" for c, f in zip(labels, d["per_class_f1"])})
            if "end_to_end" in d:
                row["e2e_acc"] = f"{d['end_to_end']['accuracy']:.3f}"
                row["e2e_macroF1"] = f"{d['end_to_end']['macro_f1']:.3f}"
            rows.append(row)
        cols = (["name", "n", "acc", "macroF1"] + [f"F1_{c}" for c in labels]
                + (["e2e_acc", "e2e_macroF1"] if rows and "e2e_acc" in rows[0] else []))
        parts.append(fmt_table(rows, cols, "\nDuration by language pair:"))
    if "duration_per_direction" in report:
        rows = [dict(name=k, n=f"{d['n']:,}", acc=f"{d['accuracy']:.3f}",
                     macroF1=f"{d['macro_f1']:.3f}")
                for k, d in report["duration_per_direction"].items()]
        parts.append(fmt_table(rows, ["name", "n", "acc", "macroF1"], "\nDuration by direction:"))
    return parts


def format_duration(report: dict, title: str) -> str:
    return "\n".join([title, *duration_parts(report)])

In [ ]:
%%writefile src/streaming.py
"""Streaming DataLoader for the anticipatory switch task.

The tagged corpus is 22.3M tokens. Materialising it as Python lists of strings
costs roughly 3 GB, so nothing here ever holds more than one Arrow batch. The
parquet is read with ``ParquetFile.iter_batches``, which decodes column pages
incrementally: a full pass over the six columns the task needs peaks at ~175 MB
RSS and takes 0.3 s.

Four things this layer has to get right:

1. **Splits must be row-level, never token-level.** Two windows from the same
   conversation share vocabulary, topic and speaker; splitting tokens would
   leak. ``row_id % 10`` assigns 0-7 train / 8 val / 9 test. The row ids are
   sequential and the source CSVs are contiguous per language, so a modulo is
   an even interleave - every language lands in every split in proportion,
   with no shuffle and no state to persist.
2. **Long rows must be windowed without amputating context.** A conversation
   longer than the window is cut into overlapping windows: the first
   ``window - stride`` positions of every continuation window are *context
   only*, fed to the model but masked out of the loss, so no position is ever
   scored on a truncated prefix.
3. **Workers must not duplicate data.** Each DataLoader worker opens its own
   handle and keeps only the Arrow batches where ``batch_index % num_workers
   == worker_id``.
4. **Shuffling must not require the whole corpus.** A bounded reservoir buffer
   shuffles the stream locally; combined with the interleaved split this is
   enough to decorrelate consecutive windows from the same conversation.
"""
from __future__ import annotations

import random

import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import IterableDataset, get_worker_info

from .anticipation import N_DUR, NONE, row_example, subword_example
from .evaluation import PAIR_KEYS, pair_key

PAD, UNK, BOS = 0, 1, 2
_SPECIALS = ["<pad>", "<unk>", "<bos>"]
PAD_LANG = 2          # padded positions must not look like first-language
assert NONE == PAD_LANG          # NONE pieces and padding share the "no language" slot

_SUBWORD_COLUMNS = ["row_id", "first_language", "second_language", "quality",
                    "piece_ids", "piece_lang", "word_end", "piece_word", "piece_utt"]
COLUMNS = {"word": None, "subword": _SUBWORD_COLUMNS}      # None = word defaults
DEFAULT_PATHS = {"word": "data/tagged.parquet", "subword": "data/subword.parquet"}
VOCAB_PATHS = {"word": "data/vocab_anticipation.json",
               "subword": "data/vocab_subword.json"}


def example_for(unit: str):
    return {"word": row_example, "subword": subword_example}[unit]

_COLUMNS = ["row_id", "tokens", "token_lang", "first_language",
            "second_language", "quality"]

SPLITS = {"train": set(range(8)), "val": {8}, "test": {9}}


def in_split(row_id: int, split: str) -> bool:
    return split == "all" or (row_id % 10) in SPLITS[split]


def iter_rows(path: str, split: str = "train", batch_rows: int = 512,
              languages=None, qualities=None, columns=None, row_group_order=None):
    """Yield one dict per row, streaming. Never holds more than one batch.

    ``row_group_order`` reads the file's row groups in that order instead of
    file order (the per-epoch reshuffle); every worker must get the same order,
    because workers shard by the running batch index.
    """
    info = get_worker_info()
    wid, nworkers = (info.id, info.num_workers) if info else (0, 1)

    pf = pq.ParquetFile(path)
    # is_entity only exists in parquets tagged after the names-policy change.
    names = pf.schema_arrow.names
    cols = [c for c in (columns or _COLUMNS + ["is_entity"]) if c in names]   # optional columns
    if row_group_order is None:
        batches = pf.iter_batches(batch_size=batch_rows, columns=cols)
    else:
        batches = (b for g in row_group_order
                   for b in pf.iter_batches(batch_size=batch_rows, row_groups=[int(g)], columns=cols))
    for bi, batch in enumerate(batches):
        if bi % nworkers != wid:
            continue
        for row in batch.to_pylist():
            if not in_split(row["row_id"], split):
                continue
            if languages and row["first_language"] not in languages:
                continue
            if qualities and row["quality"] not in qualities:
                continue
            yield row


def build_vocab(path: str, max_size: int = 50_000, split: str = "train",
                min_count: int = 2) -> dict[str, int]:
    """Word vocabulary from the TRAIN split only - val/test words stay unseen."""
    from collections import Counter

    counts: Counter[str] = Counter()
    for row in iter_rows(path, split=split):
        ex = row_example(row)
        if ex:
            counts.update(w.lower() for w in ex["words"])
    vocab = {tok: i for i, tok in enumerate(_SPECIALS)}
    for word, c in counts.most_common():
        if c < min_count or len(vocab) >= max_size:
            break
        vocab[word] = len(vocab)
    return vocab


def build_subword_vocab(path: str, max_size: int = 100_000, split: str = "train",
                        min_count: int = 2) -> dict[str, int]:
    """Compact vocabulary of XLM-R piece ids seen in the TRAIN split.

    Keys are ``str(xlmr_id)`` (JSON keys must be strings). The compact table
    keeps the from-scratch model's embedding matrix small; a pretrained XLM-R
    backbone uses the raw ids instead.
    """
    from .subword import XLMR_VOCAB
    counts = np.zeros(XLMR_VOCAB, dtype=np.int64)
    for row in iter_rows(path, split=split, columns=_SUBWORD_COLUMNS):
        counts += np.bincount(np.asarray(row["piece_ids"], dtype=np.int64),
                              minlength=XLMR_VOCAB)
    vocab = {tok: i for i, tok in enumerate(_SPECIALS)}
    for pid in np.argsort(-counts, kind="stable"):
        if counts[pid] < min_count or len(vocab) >= max_size:
            break
        vocab[str(int(pid))] = len(vocab)
    return vocab


def encode(words, vocab) -> np.ndarray:
    return np.fromiter((vocab.get(w.lower(), UNK) for w in words),
                       dtype=np.int64, count=len(words))


def encode_pieces(ids, vocab, sep_id: int | None = None) -> np.ndarray:
    """Compact ids for the from-scratch model; ``vocab=None`` keeps raw piece ids.

    Id -1 marks an inserted turn separator: the tokenizer's separator
    (``sep_id``) in raw mode, ``<bos>`` in the compact vocabulary.
    """
    if vocab is None:
        out = np.asarray(ids, dtype=np.int64).copy()
        out[out == -1] = BOS if sep_id is None else sep_id
        return out
    return np.fromiter((BOS if int(i) == -1 else vocab.get(str(int(i)), UNK) for i in ids),
                       dtype=np.int64, count=len(ids))


class SwitchWindowDataset(IterableDataset):
    """Streams fixed-length causal windows of the anticipatory task.

    Each yielded window is a dict of length-``window`` arrays. ``sw_mask``
    marks positions that are scored; a position is scored only when it has a
    defined label AND a full prefix inside this window.

    ``unit="subword"`` streams ``data/subword.parquet``: ``word_ids`` hold compact
    piece ids, and NONE pieces are unscored inputs.
    """

    def __init__(self, path: str, vocab: dict[str, int], split: str = "train",
                 window: int = 256, stride: int = 192, shuffle_buffer: int = 4096,
                 seed: int = 0, languages=None, qualities=None,
                 batch_rows: int = 512, pair_index: dict[str, int] | None = None,
                 unit: str = "word", predict_at: str = "piece", lang_input: str = "current",
                 turn_sep: bool = False, sep_id: int | None = None, reshuffle: bool = False,
                 decide_at: str = "word", blank_ids=None):
        assert predict_at in ("piece", "word_end") and lang_input in ("current", "prev_word")
        assert decide_at in ("word", "gap")
        if decide_at == "gap" and unit != "subword":
            raise ValueError("decide_at='gap' needs the sub-word labels")
        assert 0 < stride <= window
        self.path, self.vocab, self.split = path, vocab, split
        self.window, self.stride = window, stride
        self.shuffle_buffer = shuffle_buffer if split == "train" else 0
        self.seed = seed
        self.languages, self.qualities = languages, qualities
        self.batch_rows = batch_rows
        self.pair_index = pair_index or {}
        self.unit = unit
        self.predict_at, self.lang_input = predict_at, lang_input
        self.turn_sep, self.sep_id = turn_sep, sep_id
        # decide_at="gap": each label sits on the punctuation after its word
        # (anticipation.gap_landing); blank_ids are the bare space-marker ids.
        self.decide_at, self.blank_ids = decide_at, blank_ids
        # reshuffle: a new order every epoch (--reshuffle-epochs). Off, every epoch
        # repeats epoch 0's order, as all runs before 2026-11-01 did. Opt-in because a
        # 12k-step run is 1.02 epochs on 15 pairs and 1.5 on the 6 script pairs, so
        # turning it on changes those runs. The epoch is counted inside __iter__, so it
        # also advances in persistent worker copies.
        self.reshuffle = reshuffle
        self._epochs = 0

    def _windows(self, ex):
        w, s = self.window, self.stride
        ids = (encode_pieces(ex["ids"], self.vocab, self.sep_id) if self.unit == "subword"
               else encode(ex["words"], self.vocab))
        if self.lang_input == "prev_word":        # the last finished word's language
            if ex.get("lang_in") is None:
                raise ValueError("--prev-lang needs piece_word: rebuild the labels")
            langs = ex["lang_in"].astype(np.int64)
        else:
            langs = ex["langs"].astype(np.int64)
        n = len(ids)
        pair = self.pair_index.get(ex["first_language"], 0)
        report = PAIR_KEYS.index(pair_key(ex["first_language"], ex["second_language"]))
        warmup = w - s

        start = 0
        first = True
        while start < n:
            stop = min(start + w, n)
            k = stop - start
            out = {name: np.zeros(w, dtype=dt) for name, dt in
                   (("word_ids", np.int64), ("lang_ids", np.int64),
                    ("y_sw", np.int64), ("y_dur", np.int64))}
            out["sw_mask"] = np.zeros(w, dtype=bool)
            out["dur_mask"] = np.zeros(w, dtype=bool)

            out["word_ids"][:k] = ids[start:stop]
            out["lang_ids"][:] = PAD_LANG
            out["lang_ids"][:k] = langs[start:stop]
            out["cur_lang"] = np.full(w, PAD_LANG, dtype=np.int64)   # for reporting direction
            out["cur_lang"][:k] = ex.get("cur_lang", ex["langs"])[start:stop]
            out["y_sw"][:k] = np.maximum(ex["y_sw"][start:stop], 0)
            out["y_dur"][:k] = np.maximum(ex["y_dur"][start:stop], 0)

            scored = np.zeros(w, dtype=bool)
            scored[:k] = ex["y_sw"][start:stop] >= 0
            if self.predict_at == "word_end":     # only the last piece of each word
                if ex.get("unit_end") is None:
                    raise ValueError("--predict-at word_end needs piece_word: rebuild the labels")
                scored[:k] &= ex["unit_end"][start:stop]
            if not first:
                scored[:warmup] = False          # context-only warm-up
            out["sw_mask"] = scored
            dur = np.zeros(w, dtype=bool)
            dur[:k] = ex["dur_known"][start:stop]
            out["dur_mask"] = dur & scored
            sh = np.zeros(w, dtype=bool)
            if "shared" in ex:                   # hand-built examples may omit it
                sh[:k] = ex["shared"][start:stop]
            out["shared_mask"] = sh & scored     # positions comparable across tokenizers

            out["pair_id"] = np.int64(pair)
            out["report_pair"] = np.int64(report)
            out["row_id"] = np.int64(ex["row_id"])
            out["start"] = np.int64(start)        # window offset in the row: position = start + i
            yield out

            if stop == n:
                break
            start += s
            first = False

    def epoch_order(self, epoch: int):
        """Row-group order for an epoch: file order in epoch 0 (so the first epoch is
        unchanged), then a seeded permutation, the same in every worker."""
        if epoch == 0 or not self.reshuffle:
            return None
        n = pq.ParquetFile(self.path).metadata.num_row_groups
        return np.random.default_rng([self.seed, epoch]).permutation(n).tolist()

    def _stream(self, epoch: int = 0):
        make = example_for(self.unit)
        for row in iter_rows(self.path, self.split, self.batch_rows,
                             self.languages, self.qualities, columns=COLUMNS[self.unit],
                             row_group_order=self.epoch_order(epoch)):
            kw = dict(turn_sep=True) if self.turn_sep else {}
            if self.decide_at == "gap":
                kw.update(decide_at="gap", blank_ids=self.blank_ids)
            ex = make(row, **kw)
            if ex is None:
                continue
            yield from self._windows(ex)

    def __iter__(self):
        if not self.shuffle_buffer:              # val / test: always file order
            yield from self._stream()
            return
        epoch, self._epochs = self._epochs, self._epochs + 1
        if not self.reshuffle:
            epoch = 0
        stream = self._stream(epoch)
        info = get_worker_info()
        wid = info.id if info else 0
        # Epoch 0 keeps the original seed exactly; later epochs get their own buffer order.
        rng = random.Random(self.seed + wid if epoch == 0 else f"{self.seed}-{wid}-{epoch}")
        buf = []
        for item in stream:
            buf.append(item)
            if len(buf) >= self.shuffle_buffer:
                j = rng.randrange(len(buf))
                buf[j], buf[-1] = buf[-1], buf[j]
                yield buf.pop()
        rng.shuffle(buf)
        yield from buf


def collate(batch):
    out = {}
    for key in ("word_ids", "lang_ids", "cur_lang", "y_sw", "y_dur"):
        out[key] = torch.from_numpy(np.stack([b[key] for b in batch]))
    for key in ("sw_mask", "dur_mask", "shared_mask"):
        out[key] = torch.from_numpy(np.stack([b[key] for b in batch]))
    for key in ("pair_id", "report_pair", "row_id", "start"):
        out[key] = torch.tensor([int(b[key]) for b in batch])
    return out


def make_loader(path, vocab, split="train", batch_size=32, num_workers=0, **kw):
    ds = SwitchWindowDataset(path, vocab, split=split, **kw)
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, num_workers=num_workers, collate_fn=collate,
        persistent_workers=bool(num_workers),
    )

In [ ]:
%%writefile src/causal_model.py
"""A prefix-only (causal) transformer for anticipatory switch prediction.

The whole task is defined by what the model is NOT allowed to see. At position
``t`` it predicts properties of token ``t+1``, so any path by which information
from ``t+1..n`` reaches position ``t`` silently turns the benchmark into a
tagging task with a near-perfect ceiling. Three such paths exist, and all three
are closed here:

1. **Attention.** ``nn.TransformerEncoderLayer`` is bidirectional by default.
   An additive float mask with ``-inf`` above the diagonal, passed with
   ``is_causal=True``, restricts every query at ``t`` to keys ``<= t``. The
   flag alone is only a hint to the fast path - the explicit mask is what
   enforces it, so both are supplied.
2. **Normalisation.** Any layer that pools across time leaks the future.
   ``LayerNorm`` normalises over the feature axis only, which is safe;
   ``BatchNorm1d`` over time would not be, and is not used.
3. **Labels as inputs.** The input at ``t`` carries the language of token
   ``t``, which is in the prefix and therefore legal. It never carries
   ``lang(t+1)``, which IS the label. The two are built from the same array,
   one shifted against the other, so the shift is the only thing standing
   between a legitimate model and a 100% score - ``check_causality`` verifies
   it empirically rather than by inspection. In sub-word mode there is no
   language input at all (``use_lang=False``), because at a mid-word piece the
   word's label depends on pieces not yet read.

``check_causality`` perturbs every token after position ``t`` and asserts the
logits at ``t`` are bit-identical. That is a stronger statement than "we set
the mask": it tests the assembled model, including the heads.
"""
from __future__ import annotations

import torch
import torch.nn as nn

from .anticipation import N_DUR


DUR_HEADS = ("softmax", "ordinal")


class OrdinalHead(nn.Module):
    """Cumulative-link duration head: one logit per cut, ``P(class > k)``.

    Small < Medium < Large is ordered, and a softmax head ignores that. This
    head scores one shared projection ``s = w.h`` against ``n_classes - 1``
    cut points (CORAL-style), so the classes are the intervals between cuts.
    The cuts are kept in order by construction: each one is the last minus a
    softplus gap, so ``P(> Small) >= P(> Medium)`` always holds and the class
    probabilities from ``duration_probs`` are never negative.
    """

    def __init__(self, d: int, n_classes: int = N_DUR):
        super().__init__()
        self.proj = nn.Linear(d, 1)
        # Start near the corpus shares (about 25 / 25 / 50): P(>Small) 0.75, P(>Medium) 0.5.
        self.first = nn.Parameter(torch.tensor(1.1))
        self.gaps = nn.Parameter(torch.full((n_classes - 2,), 0.7))

    def forward(self, h):
        offs = torch.cat([self.first.reshape(1),
                          self.first - torch.cumsum(nn.functional.softplus(self.gaps), 0)])
        return self.proj(h) + offs


def make_dur_head(d: int, n_dur: int = N_DUR, kind: str = "softmax") -> nn.Module:
    if kind not in DUR_HEADS:
        raise ValueError(f"dur_head is one of {DUR_HEADS}")
    return OrdinalHead(d, n_dur) if kind == "ordinal" else nn.Linear(d, n_dur)


def duration_probs(dur, ordinal: bool):
    """Class probabilities from the duration head's output, either kind."""
    if not ordinal:
        return torch.softmax(dur, -1)
    c = torch.sigmoid(dur)                                  # [..., n-1]: P(class > k)
    one, zero = torch.ones_like(c[..., :1]), torch.zeros_like(c[..., :1])
    upper = torch.cat([one, c], -1)
    return (upper - torch.cat([c, zero], -1)).clamp(min=0)


def ordinal_targets(y_dur, n_dur: int = N_DUR):
    """``[y > 0, y > 1, ...]`` as floats: the cumulative targets of ``OrdinalHead``."""
    k = torch.arange(n_dur - 1, device=y_dur.device)
    return (y_dur.unsqueeze(-1) > k).float()


class CausalSwitchModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, n_layers: int = 4,
                 n_heads: int = 4, d_ff: int = 512, max_len: int = 512,
                 dropout: float = 0.1, n_pairs: int = 16, use_pair: bool = False,
                 use_lang: bool = True, n_dur: int = N_DUR, dur_head: str = "softmax"):
        super().__init__()
        self.d_model = d_model
        self.dur_ordinal = dur_head == "ordinal"
        self.use_pair = use_pair
        self.word_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # 2 real slots + 1 pad slot; the language of the CURRENT token only.
        # Sub-word models take no language input (use_lang=False): a word's
        # label is known only once all of its pieces have been read.
        self.lang_emb = nn.Embedding(3, d_model) if use_lang else None
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.pair_emb = nn.Embedding(n_pairs, d_model) if use_pair else None
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, n_layers,
                                             norm=nn.LayerNorm(d_model),
                                             enable_nested_tensor=False)
        self.head_sw = nn.Linear(d_model, 1)
        self.head_dur = make_dur_head(d_model, n_dur, dur_head)

    def forward(self, word_ids, lang_ids=None, pair_id=None):
        b, t = word_ids.shape
        pos = torch.arange(t, device=word_ids.device).unsqueeze(0)
        x = self.word_emb(word_ids) + self.pos_emb(pos)
        if self.lang_emb is not None:
            # lang slot 2 is reserved for padding so pads never look like L1.
            x = x + self.lang_emb(lang_ids)
        if self.pair_emb is not None and pair_id is not None:
            x = x + self.pair_emb(pair_id).unsqueeze(1)
        x = self.drop(x)

        mask = torch.triu(
            torch.full((t, t), float("-inf"), device=word_ids.device), diagonal=1
        )
        h = self.encoder(x, mask=mask, is_causal=True)
        return self.head_sw(h).squeeze(-1), self.head_dur(h)


@torch.no_grad()
def check_causality(model, vocab_size: int, seq_len: int = 32,
                    device: str = "cpu", tol: float = 0.0) -> dict:
    """Empirically verify prefix-only context.

    Builds one sequence, then rewrites every position after ``t`` with random
    tokens and languages. If any logit at or before ``t`` moves, the future is
    reaching the past.
    """
    model.eval().to(device)
    g = torch.Generator(device="cpu").manual_seed(0)
    words = torch.randint(3, vocab_size, (1, seq_len), generator=g).to(device)
    langs = torch.randint(0, 2, (1, seq_len), generator=g).to(device)
    sw_a, dur_a = model(words, langs)

    worst = 0.0
    for t in range(seq_len - 1):
        w2, l2 = words.clone(), langs.clone()
        w2[0, t + 1:] = torch.randint(3, vocab_size, (seq_len - t - 1,), generator=g)
        l2[0, t + 1:] = torch.randint(0, 2, (seq_len - t - 1,), generator=g)
        sw_b, dur_b = model(w2, l2)
        worst = max(worst,
                    float((sw_a[0, :t + 1] - sw_b[0, :t + 1]).abs().max()),
                    float((dur_a[0, :t + 1] - dur_b[0, :t + 1]).abs().max()))
    return dict(max_abs_logit_drift=worst, causal=worst <= tol,
                positions_tested=seq_len - 1)


@torch.no_grad()
def check_context_free(model, vocab_size: int, seq_len: int = 32,
                       device: str = "cpu", tol: float = 0.0) -> dict:
    """Verify that the prediction at ``t`` reads token ``t`` and nothing else.

    The current-token model (``context=1``) passes ``check_causality`` trivially,
    so this rewrites every *other* position, past and future, and asserts that
    no logit at ``t`` moves. A full-prefix model fails it.
    """
    model.eval().to(device)
    g = torch.Generator(device="cpu").manual_seed(0)
    words = torch.randint(3, vocab_size, (1, seq_len), generator=g).to(device)
    langs = torch.randint(0, 2, (1, seq_len), generator=g).to(device)
    sw_a, dur_a = model(words, langs)

    worst = 0.0
    for t in range(seq_len):
        w2 = torch.randint(3, vocab_size, (1, seq_len), generator=g).to(device)
        l2 = torch.randint(0, 2, (1, seq_len), generator=g).to(device)
        w2[0, t], l2[0, t] = words[0, t], langs[0, t]
        sw_b, dur_b = model(w2, l2)
        worst = max(worst, float((sw_a[0, t] - sw_b[0, t]).abs()),
                    float((dur_a[0, t] - dur_b[0, t]).abs().max()))
    return dict(max_abs_logit_drift=worst, context_free=worst <= tol, positions_tested=seq_len)

In [ ]:
%%writefile src/lora.py
"""Low-rank adapters (LoRA, Hu et al. 2021) for the pretrained encoders, without ``peft``.

Each targeted ``nn.Linear`` keeps its pretrained weight W frozen and learns a
rank-``r`` update: ``y = W x + (alpha / r) * B A x``. ``B`` starts at zero, so
the adapted encoder is exactly the pretrained one at step 0, like the
zero-initialised language and pair embeddings.

``apply_lora`` freezes the whole encoder first, so only the adapters (plus the
model's own new parameters outside the encoder: heads, language and pair
embeddings) train. LoRA only changes linear layers inside a token's own
computation, so the causal mask is untouched; ``check_causality`` still
re-verifies the assembled model.
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn

# XLM-R / RoBERTa / BERT names: attention query/key/value, the attention output
# projection and both feed-forward layers are all called one of these.
DEFAULT_TARGETS = ("query", "key", "value", "dense")


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        if r <= 0:
            raise ValueError("LoRA rank must be positive")
        self.base = base
        self.r, self.scale = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.t() @ self.lora_B.t()) * self.scale


def apply_lora(encoder: nn.Module, r: int = 16, alpha: float | None = None, dropout: float = 0.05,
               targets=DEFAULT_TARGETS) -> int:
    """Freeze ``encoder`` and wrap its targeted linear layers; returns how many were wrapped."""
    for p in encoder.parameters():
        p.requires_grad_(False)
    alpha = float(2 * r if alpha is None else alpha)
    wrapped = 0
    for parent in list(encoder.modules()):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and name in targets:
                setattr(parent, name, LoRALinear(child, r, alpha, dropout))
                wrapped += 1
    if not wrapped:
        raise ValueError(f"no linear layer named any of {targets}")
    return wrapped


def is_lora_param(name: str) -> bool:
    return ".lora_A" in name or ".lora_B" in name

In [ ]:
%%writefile src/xlmr_model.py
"""Pretrained encoders with a causal mask: XLM-R, RoBERTa and mBERT.

These encoders were pretrained bidirectionally. ``is_decoder=True`` makes
Hugging Face apply a lower-triangular (causal) mask in every self-attention
layer, so the representation at piece t is built from pieces 1..t only. We
measured this (earlier positions move by 0.00 when later pieces change), and
``check_causality`` re-verifies it on the assembled model before any training.

| key       | checkpoint                     | pretrained on             | tokenizer          |
|-----------|--------------------------------|---------------------------|--------------------|
| "xlmr"    | xlm-roberta-base               | 100 languages (CC-100)    | SentencePiece 250K |
| "roberta" | roberta-base                   | English only              | byte-level BPE 50K |
| "mbert"   | bert-base-multilingual-cased   | 104 languages (Wikipedia) | WordPiece 119K     |

The interface matches ``CausalSwitchModel``: ``forward(word_ids, lang_ids=None,
pair_id=None) -> (switch logits [b, t], duration logits [b, t, n_dur])``. By
default the language and pair inputs are accepted and ignored, and the model
reads the text only. ``use_lang`` (``--prev-lang``) adds an embedding of the
last finished word's language; ``use_pair`` (``--pair-emb``) adds an embedding of
the row's language pair to every piece. Both start at zero, so step 0 is the
pretrained model, and a pair never seen in training keeps a zero embedding (no
pair signal) rather than a random one. ``lora_r`` > 0 freezes the encoder and
trains low-rank adapters instead (``src/lora.py``). Inputs are the backbone's raw piece ids, with no special
tokens. Windows are right-padded, and under a causal mask a real position
never attends to padding after it, so no attention mask is needed.

``context=1`` is the current-token-only model, the pretrained counterpart of
the ``naive_l2`` baseline. Every piece is encoded as its own length-1
sequence, so the prediction at t reads piece t and nothing else (not even the
past), and every piece gets the same position. In eval mode the model is then a
function of the piece id alone.
"""
from __future__ import annotations

import torch.nn as nn

from .anticipation import N_DUR
from .causal_model import make_dur_head

BACKBONES = {"xlmr": "xlm-roberta-base", "roberta": "roberta-base",
             "mbert": "bert-base-multilingual-cased"}


class CausalPretrained(nn.Module):
    def __init__(self, name: str = "xlm-roberta-base", config=None, pretrained: bool = True,
                 n_dur: int = N_DUR, dropout: float = 0.1, use_lang: bool = False,
                 context: int | None = None, dur_head: str = "softmax",
                 use_pair: bool = False, n_pairs: int = 0, lora_r: int = 0,
                 lora_alpha: float | None = None):
        super().__init__()
        self.dur_ordinal = dur_head == "ordinal"
        if context not in (None, 1):
            raise ValueError("context is None (the whole prefix) or 1 (the current piece only)")
        if context == 1 and use_lang:
            raise ValueError("a current-token model takes no language input: it would be context")
        if use_pair and n_pairs <= 0:
            raise ValueError("use_pair needs n_pairs")
        self.context = context
        from transformers import AutoConfig, AutoModel

        causal = dict(is_decoder=True, add_cross_attention=False)
        if config is not None:                   # explicit config: random init (tests)
            for k, v in causal.items():
                setattr(config, k, v)
            self.encoder = AutoModel.from_config(config, add_pooling_layer=False)
        elif pretrained:                         # fine-tuning: the pretrained weights
            self.encoder = AutoModel.from_pretrained(name, add_pooling_layer=False, **causal)
        else:                                    # architecture only, to load a checkpoint into
            cfg = AutoConfig.from_pretrained(name, **causal)
            self.encoder = AutoModel.from_config(cfg, add_pooling_layer=False)
        assert self.encoder.config.is_decoder, "the encoder must run with a causal mask"
        self.vocab_size = self.encoder.config.vocab_size
        d = self.encoder.config.hidden_size
        # Optional language input (--prev-lang): a learned embedding for the language of
        # the last finished word (slots L1, L2, none), added to the piece embeddings.
        # Initialised at zero, so training starts from the pretrained model unchanged.
        self.lang_emb = nn.Embedding(3, d) if use_lang else None
        self.pair_emb = nn.Embedding(n_pairs, d) if use_pair else None
        if use_pair:
            self.pair_key = "report_pair"        # the 15-pair index (see models.pair_ids)
        for emb in (self.lang_emb, self.pair_emb):
            if emb is not None:
                nn.init.zeros_(emb.weight)
        self.lora_r = lora_r
        if lora_r:
            from .lora import apply_lora
            apply_lora(self.encoder, r=lora_r, alpha=lora_alpha)
        self.drop = nn.Dropout(dropout)
        self.head_sw = nn.Linear(d, 1)
        self.head_dur = make_dur_head(d, n_dur, dur_head)

    def forward(self, word_ids, lang_ids=None, pair_id=None):
        if self.context == 1:                    # [b, t] -> [b*t, 1]: one piece per sequence
            b, t = word_ids.shape
            pair = None if pair_id is None else pair_id.repeat_interleave(t)
            sw, dur = self._encode(word_ids.reshape(b * t, 1), None, pair)
            return sw.reshape(b, t), dur.reshape(b, t, -1)
        return self._encode(word_ids, lang_ids, pair_id)

    def _encode(self, word_ids, lang_ids, pair_id=None):
        # use_cache=False: a decoder-mode encoder would otherwise build a key/value
        # cache on every call, which only costs memory here.
        add_pair = self.pair_emb is not None and pair_id is not None
        if self.lang_emb is None and not add_pair:
            h = self.encoder(input_ids=word_ids, use_cache=False).last_hidden_state
        else:
            emb = self.encoder.get_input_embeddings()(word_ids)
            if self.lang_emb is not None:
                emb = emb + self.lang_emb(lang_ids)
            if add_pair:        # one pair per row: the same vector at every position
                emb = emb + self.pair_emb(pair_id).unsqueeze(1)
            h = self.encoder(inputs_embeds=emb, use_cache=False).last_hidden_state
        h = self.drop(h)
        return self.head_sw(h).squeeze(-1), self.head_dur(h)


class CausalXLMR(CausalPretrained):
    """The XLM-R instance (kept for existing checkpoints and tests)."""

    def __init__(self, name: str = "xlm-roberta-base", **kw):
        super().__init__(name, **kw)

In [ ]:
%%writefile src/models.py
"""Build any anticipation model from a run's settings (training or a checkpoint's args)."""
from __future__ import annotations

from .anticipation import N_DUR
from .causal_model import CausalSwitchModel
from .evaluation import PAIR_KEYS

PRETRAINED = ("xlmr", "roberta", "mbert")


def build_model(cfg: dict, vocab_size: int, n_pairs: int = 14, use_pair: bool = False,
                use_lang: bool = True, n_dur: int = N_DUR, pretrained: bool = True,
                xlmr_config=None):
    """``cfg["model"]`` is "scratch" (the from-scratch transformer), or a pretrained
    backbone: "xlmr", "roberta" or "mbert" (see ``xlmr_model.BACKBONES``)."""
    model = cfg.get("model", "scratch")
    if model in PRETRAINED:
        from .xlmr_model import BACKBONES, CausalPretrained, CausalXLMR
        name = cfg.get("backbone") or (cfg.get("xlmr_name") if model == "xlmr" else None) \
            or BACKBONES[model]
        cls = CausalXLMR if model == "xlmr" else CausalPretrained
        return cls(name, config=xlmr_config, pretrained=pretrained, n_dur=n_dur,
                   use_lang=cfg.get("prev_lang", False), context=cfg.get("context"),
                   dur_head=cfg.get("dur_head", "softmax"),
                   use_pair=bool(cfg.get("pair_emb", False)), n_pairs=len(PAIR_KEYS),
                   lora_r=int(cfg.get("lora_r") or 0), lora_alpha=cfg.get("lora_alpha"))
    return CausalSwitchModel(vocab_size, d_model=cfg["d_model"], n_layers=cfg["layers"],
                             n_heads=cfg["heads"], d_ff=cfg["d_model"] * 2,
                             max_len=cfg["window"], n_pairs=n_pairs, use_pair=use_pair,
                             use_lang=use_lang, n_dur=n_dur,
                             dur_head=cfg.get("dur_head", "softmax"))


def pair_ids(model, batch):
    """The pair input a model expects from a loader batch.

    The from-scratch model's ``pair_id`` indexes first languages (14), so
    German->English and German->French share one id. A pretrained model with
    ``--pair-emb`` reads ``report_pair``, the full 15-pair index (``PAIR_KEYS``).
    """
    return batch[getattr(model, "pair_key", "pair_id")]


def load_checkpoint(path: str, map_location="cpu", fallback_args: dict | None = None):
    """``(model in eval mode, checkpoint dict)`` from a ``train_anticipation.py --ckpt`` file.

    The architecture is rebuilt from the checkpoint's own args (a pretrained backbone's
    config only, no weight download), then the trained weights are loaded.
    """
    import torch
    ck = torch.load(path, map_location=map_location, weights_only=False)
    cfg = {**(fallback_args or {}), **ck.get("args", {})}
    model = build_model(cfg, ck.get("vocab_size"), use_lang=ck.get("use_lang", True),
                        n_dur=int(cfg.get("dur_classes") or N_DUR), pretrained=False)
    model.load_state_dict(ck["state"])
    model.eval()
    ck["args"] = cfg
    return model, ck

In [ ]:
%%writefile src/training.py
"""Training helpers: reproducible seeds, class-weighted duration loss, the evaluation
loop and its scoring (shared by train_anticipation.py and eval_lince.py), saved predictions."""
from __future__ import annotations

import contextlib
import random

import numpy as np

from .anticipation import N_DUR, collapse_duration
from .causal_model import duration_probs
from .evaluation import full_report
from .metrics import prf
from .models import pair_ids
from .streaming import COLUMNS, example_for, iter_rows


def set_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch (model initialisation and dropout)."""
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def count_duration_classes(path: str, unit: str = "subword", languages=None,
                           predict_at: str = "piece") -> np.ndarray:
    """How often each duration class is a training target, on the TRAIN split only.

    Counts the positions the duration loss actually uses: a switch whose class
    is certain (``dur_known``), and with ``predict_at="word_end"`` only the
    last piece of each word.
    """
    make = example_for(unit)
    counts = np.zeros(N_DUR, dtype=np.int64)
    for row in iter_rows(path, split="train", languages=languages, columns=COLUMNS[unit]):
        ex = make(row)
        if ex is None:
            continue
        m = ex["dur_known"]
        if predict_at == "word_end" and ex.get("unit_end") is not None:
            m = m & ex["unit_end"]
        counts += np.bincount(ex["y_dur"][m].astype(np.int64), minlength=N_DUR)
    return counts


def duration_class_weights(counts) -> np.ndarray:
    """Inverse-frequency weights, rescaled to mean 1 (an empty class gets weight 0)."""
    counts = np.asarray(counts, dtype=float)
    w = np.where(counts > 0, 1.0 / np.maximum(counts, 1), 0.0)
    return w * (len(w) / w.sum()) if w.sum() else np.ones(len(w))


def save_predictions(path: str, val: dict, test: dict) -> None:
    """Every scored position's switch logit and labels, for offline threshold and
    calibration experiments; ``dprob`` is the duration head's softmax."""
    arrays = {f"{split}_{k}": np.asarray(v) for split, res in (("val", val), ("test", test))
              for k, v in res.items()}
    np.savez_compressed(path, **arrays)


# --------------------------------------------------------------------------- evaluation
def autocast(dev):
    """fp16 on CUDA (a T4 has no bf16); full precision elsewhere."""
    import torch
    if dev.type == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def scored_positions(b):
    """``(row_id, position in the row)`` of every scored position in a batch, in order."""
    import torch
    m = b["sw_mask"]
    row = b["row_id"].unsqueeze(1).expand_as(m)[m]
    pos = (b["start"].unsqueeze(1) + torch.arange(m.shape[1])).expand_as(m)[m]
    return row.numpy(), pos.numpy()


def head_classes(model, dur) -> int:
    """Duration classes a model predicts: a softmax head's width, or cuts + 1 (ordinal)."""
    return dur.shape[-1] + (1 if getattr(model, "dur_ordinal", False) else 0)


def batch_dur_labels(b, n_dur: int):
    """The batch's ``y_dur`` for an ``n_dur``-class head: as stored (3), or Short / Large (2)."""
    return collapse_duration(b["y_dur"]) if n_dur == 2 else b["y_dur"]


def evaluate(model, loader, dev, max_batches=0):
    """Logits and labels at every scored position, with pair and direction."""
    import torch
    with torch.no_grad():
        return _evaluate(model, loader, dev, max_batches)


def _evaluate(model, loader, dev, max_batches=0):
    import torch
    model.eval()
    out = {k: [] for k in ("logit", "y", "pair", "lang", "dtrue", "dpred", "dprob", "dpair",
                           "dlang", "dlogit", "drow", "shared", "row", "pos")}
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with autocast(dev):
            sw, dur = model(b["word_ids"].to(dev), b["lang_ids"].to(dev), pair_ids(model, b).to(dev))
        m = b["sw_mask"]
        out["logit"].append(sw.float().cpu()[m].numpy())
        out["y"].append(b["y_sw"][m].numpy())
        out["pair"].append(b["report_pair"].unsqueeze(1).expand_as(m)[m].numpy())
        out["lang"].append(b["cur_lang"][m].numpy())        # direction: current language
        dm = b["dur_mask"]
        if dm.any():
            out["dtrue"].append(batch_dur_labels(b, head_classes(model, dur))[dm].numpy())
            probs = duration_probs(dur.float().cpu()[dm], getattr(model, "dur_ordinal", False))
            out["dpred"].append(probs.argmax(-1).numpy())
            out["dprob"].append(probs.numpy())
            # aligned with dtrue: dur_mask is a subset of sw_mask
            out["dpair"].append(b["report_pair"].unsqueeze(1).expand_as(dm)[dm].numpy())
            out["dlang"].append(b["cur_lang"][dm].numpy())
            out["dlogit"].append(sw.float().cpu()[dm].numpy())
            out["drow"].append(b["row_id"].unsqueeze(1).expand_as(dm)[dm].numpy())
        out["shared"].append(b["shared_mask"][m].numpy())
        row, pos = scored_positions(b)                      # trace errors back to the text
        out["row"].append(row)
        out["pos"].append(pos)
    return {k: np.concatenate(v) if v else np.empty(0) for k, v in out.items()}


def tune_threshold(prob, truth):
    """Pick the F1-maximising threshold on a held-out split."""
    best = (0.0, 0.5)
    for t in np.quantile(prob, np.linspace(0.50, 0.999, 60)):
        best = max(best, (prf(truth, prob >= t)["f1"], float(t)))
    return best[1], best[0]


def score(res, tau):
    """The full report, plus ``shared``: the same report on the positions every
    model shares (the last piece of each non-CJK word), comparable across tokenizers."""
    pred = (1 / (1 + np.exp(-res["logit"])) >= tau).astype(int)
    # Duration side arrays are aligned with dtrue; older prediction files lack them.
    def aligned(k):
        return res[k] if k in res and len(res[k]) == len(res["dtrue"]) else None

    n_dur = res["dprob"].shape[1] if np.ndim(res.get("dprob")) == 2 else N_DUR
    dlogit = aligned("dlogit")
    fired = None if dlogit is None else 1 / (1 + np.exp(-np.asarray(dlogit))) >= tau
    rep = full_report(res["y"], pred, res["pair"], res["lang"], res["dtrue"], res["dpred"],
                      dur_pairs=aligned("dpair"), dur_langs=aligned("dlang"), dur_fired=fired,
                      n_dur=n_dur)
    sh = np.asarray(res.get("shared", np.zeros(len(pred), bool)), dtype=bool)
    if sh.any():
        rep["shared"] = full_report(res["y"][sh], pred[sh], res["pair"][sh], res["lang"][sh])
    return rep

In [ ]:
%%writefile src/lince.py
"""LinCE as an out-of-domain test set for the anticipatory task.

LinCE (https://ritual.uh.edu/lince/) has tweets with **human** word-level language tags.
Two of its pairs are SwitchLingua pairs: ``spaeng`` (Spanish->English) and ``hineng``
(romanized Hindi->English). The tags are gold, so the switch and duration labels come
straight from them, with no tagger involved. The tweets only need to be cut into this
project's tokens and then into each model's pieces (``subword.build_frame``), so the
evaluation matches SwitchLingua's exactly.

Tag mapping (LinCE: ``lang1`` = English, ``lang2`` = the other language):

* ``lang1`` -> English (L2), ``lang2`` -> Spanish / Hindi (L1).
* ``ne``, ``mixed``, ``ambiguous``, ``fw``, ``unk`` and lettered ``other`` (@handles,
  #hashtags, URLs) -> ``NE``. That is a language-less word: it neither breaks nor
  extends a run. This is LinCE's own convention and our ``transparent`` names
  policy. ``absorb_other`` would otherwise fold a handle into its neighbour's run.
* Tokens with no letters (punctuation, emoji) -> ``other``, as in the corpus.

The ``validation`` and ``train`` splits both have public labels, and no model here was
trained on LinCE, so both are fair test data. The ``test`` labels are hidden.
"""
from __future__ import annotations

import os
import urllib.request

import numpy as np
import pandas as pd

from .anticipation import is_word
from .segments import NE
from .tokenizer import tokenize_mixed

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PUBLIC = os.path.join(ROOT, "data", "public")
LINCE_DIR = os.path.join(ROOT, "data", "lince")
URL = ("https://huggingface.co/datasets/lince-benchmark/lince/resolve/"
       "refs%2Fconvert%2Fparquet/lid_{pair}/{split}/0000.parquet")
PAIRS = {"spaeng": "Spanish", "hineng": "Hindi"}          # LinCE config -> first_language
SPLITS = ("validation", "train")
# row ids far above SwitchLingua's 234,172: pair block + split block + tweet index
_PAIR_BASE = {"spaeng": 10_000_000, "hineng": 20_000_000}
_SPLIT_BASE = {"validation": 0, "train": 1_000_000}


def row_id(pair: str, split: str, idx: int) -> int:
    return _PAIR_BASE[pair] + _SPLIT_BASE[split] + int(idx)


def describe_row(rid: int) -> tuple[str, str, int]:
    """``(pair, split, index)`` of a LinCE row id."""
    pair = next(p for p, b in _PAIR_BASE.items() if b <= rid < b + 10_000_000)
    rest = rid - _PAIR_BASE[pair]
    split = next(s for s, b in sorted(_SPLIT_BASE.items(), key=lambda kv: -kv[1]) if rest >= b)
    return pair, split, rest - _SPLIT_BASE[split]


def load(pair: str, split: str) -> pd.DataFrame:
    """A LinCE LID split (``idx``, ``words``, ``lid``), downloaded once into data/public/."""
    path = os.path.join(PUBLIC, f"lid_{pair}_{split}.parquet")
    if not os.path.exists(path):
        os.makedirs(PUBLIC, exist_ok=True)
        print(f"downloading LinCE lid_{pair}/{split}", flush=True)
        urllib.request.urlretrieve(URL.format(pair=pair, split=split), path)
    return pd.read_parquet(path)


def tweet_tokens(words, lid, first_language: str) -> tuple[str, list[str], list[str]]:
    """``(text, tokens, token_lang)`` for one tweet.

    The text is the LinCE words joined by spaces. Each project token takes the gold
    tag of the LinCE word its characters fall in.
    """
    words = [str(w) for w in words]
    text = " ".join(words)
    owner = np.full(len(text), -1, dtype=np.int64)          # character -> LinCE word
    pos = 0
    for k, w in enumerate(words):
        owner[pos:pos + len(w)] = k
        pos += len(w) + 1
    tokens, langs, at = [], [], 0
    for t in tokenize_mixed(text):
        s = text.find(t, at)
        if s < 0:
            raise ValueError(f"token {t!r} not found in {text!r}")
        at = s + len(t)
        tag = str(lid[owner[s]]) if owner[s] >= 0 else "other"
        tokens.append(t)
        langs.append(map_tag(tag, t, first_language))
    return text, tokens, langs


def map_tag(tag: str, token: str, first_language: str) -> str:
    if not is_word(token):
        return "other"                       # punctuation, digits, emoji: as in the corpus
    if tag == "lang1":
        return "English"
    if tag == "lang2":
        return first_language
    return NE                                # ne / mixed / ambiguous / fw / unk / @ # URLs


def to_tagged(df: pd.DataFrame, pair: str, split: str) -> tuple[pd.DataFrame, dict]:
    """A tagged-like frame plus ``texts``: the input ``subword.build_frame`` takes."""
    l1 = PAIRS[pair]
    rows, texts = [], {}
    for i, (words, lid) in enumerate(zip(df["words"], df["lid"])):
        if len(words) == 0:
            continue
        text, tokens, langs = tweet_tokens(words, lid, l1)
        rid = row_id(pair, split, int(df["idx"].iloc[i]) if "idx" in df else i)
        rows.append(dict(row_id=rid, first_language=l1, second_language="English",
                         quality="gold", tokens=tokens, token_lang=langs))
        texts[rid] = [text]
    return pd.DataFrame(rows), texts


def parquet_path(backbone: str, pair: str, split: str) -> str:
    safe = backbone.replace("/", "_")
    return os.path.join(LINCE_DIR, f"subword_{safe}_{pair}_{split}.parquet")


def build(pair: str, split: str, backbone: str, rebuild: bool = False) -> tuple[str, dict]:
    """Write (or reuse) the LinCE piece-level parquet for one pair, split and tokenizer."""
    from transformers import AutoTokenizer

    from .subword import build_frame, write_parquet
    path = parquet_path(backbone, pair, split)
    if os.path.exists(path) and not rebuild:
        return path, {}
    frame, texts = to_tagged(load(pair, split), pair, split)
    tok = AutoTokenizer.from_pretrained(backbone)
    pieces, stats = build_frame(frame, texts, tok, max_fail=0.01)
    os.makedirs(LINCE_DIR, exist_ok=True)
    write_parquet(pieces, path)
    stats["switches"] = int(sum((np.asarray(y) == 1).sum() for y in pieces.y_sw))
    stats["tweets_with_switch"] = int(sum((np.asarray(y) == 1).any() for y in pieces.y_sw))
    return path, stats


def mix_path(weight: int, data_dir: str = os.path.join(ROOT, "data")) -> str:
    return os.path.join(data_dir, f"subword_mix_lince{weight}.parquet")


def mix(sl_path: str, backbone: str, weight: int, out: str | None = None, seed: int = 0,
        pairs=tuple(PAIRS), lince_paths: dict | None = None) -> tuple[str, dict]:
    """SwitchLingua pieces + LinCE **train** tweets, for multi-dataset training.

    LinCE *validation* is never included: it stays the held-out LinCE test. The
    streaming split (``row_id % 10``) also divides LinCE train 80/10/10, so only
    its training rows (``row_id % 10 < 8``) are repeated ``weight`` times. Its val
    and test rows appear once, and duplicates never cross a split. The rows are
    shuffled (``seed``) so streaming is not a curriculum by dataset.

    ``lince_paths`` maps pair -> an already-built LinCE train parquet (tests).
    Returns ``(path, stats)``; ``stats`` gives rows and pieces per source and the
    LinCE share of training-split pieces.
    """
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    if weight < 1:
        raise ValueError("weight must be >= 1 (0 means: do not mix)")
    out = out or mix_path(weight)
    sl = pq.read_table(sl_path)
    parts, stats = [sl], {"weight": int(weight)}

    def n_pieces(t):
        return int(pc.sum(pc.list_value_length(t["piece_ids"])).as_py() or 0)

    def train_rows(t):                              # streaming split: row_id % 10 in 0..7
        return pa.array(t["row_id"].to_numpy() % 10 < 8)

    sl_train = sl.filter(train_rows(sl))
    stats["switchlingua"] = dict(rows=sl.num_rows, train_pieces=n_pieces(sl_train))
    lince_train_pieces = 0
    for pair in pairs:
        path = (lince_paths or {}).get(pair) or build(pair, "train", backbone)[0]
        t = pq.read_table(path).cast(sl.schema)
        if (pc.min(t["row_id"]).as_py() - _PAIR_BASE[pair]) < _SPLIT_BASE["train"]:
            raise ValueError(f"{path} holds LinCE validation rows; only train may be mixed")
        tr = t.filter(train_rows(t))
        parts += [t] + [tr] * (weight - 1)          # val/test rows once, train rows x weight
        lince_train_pieces += weight * n_pieces(tr)
        stats[pair] = dict(rows=t.num_rows, train_rows=tr.num_rows, rows_written=t.num_rows + (weight - 1) * tr.num_rows)
    table = pa.concat_tables(parts)
    perm = np.random.default_rng(seed).permutation(table.num_rows)
    table = table.take(pa.array(perm))
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    pq.write_table(table, out, compression="snappy", row_group_size=8192)
    stats["rows"] = table.num_rows
    stats["lince_share_of_train_pieces"] = lince_train_pieces / (lince_train_pieces + stats["switchlingua"]["train_pieces"])
    return out, stats

In [ ]:
%%writefile scripts/training/train_anticipation.py
#!/usr/bin/env python
"""Train an anticipation model and score it with the shared evaluation report.

    # the from-scratch causal transformer (words, or XLM-R pieces)
    python scripts/training/train_anticipation.py --build-vocab --path data/tagged_shuffled.parquet --steps 6000
    python scripts/training/train_anticipation.py --unit subword --build-vocab --steps 6000
    # XLM-R with a causal mask (needs a GPU: see kaggle/train_xlmr.ipynb)
    python scripts/training/train_anticipation.py --model xlmr --unit subword --steps 12000 --batch-size 16

Protocol (identical for every model): train on the train split; tune the
switch threshold on val; score test once with ``src.evaluation.full_report``
(switch P/R/F1 per pair, macro-F1, per direction; 3-class duration at true
switch points). ``check_causality`` must pass before training starts.
Everything is streamed; the process never holds more than a shuffle buffer.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

from src.anticipation import N_DUR, dur_labels  # noqa: E402
from src.causal_model import check_causality, check_context_free, ordinal_targets  # noqa: E402
from src.evaluation import format_duration, format_report  # noqa: E402
from src.metrics import apply_class_offsets, duration_scores, tune_class_offsets  # noqa: E402
from src.models import build_model, pair_ids  # noqa: E402
from src.streaming import (DEFAULT_PATHS, VOCAB_PATHS, build_subword_vocab,  # noqa: E402
                           build_vocab, make_loader)
from src.training import (autocast, batch_dur_labels, count_duration_classes,  # noqa: E402
                          duration_class_weights, evaluate, head_classes, save_predictions,
                          score, scored_positions, set_seed, tune_threshold)

UNIT_DEFAULTS = {"word": dict(window=256, stride=192, vocab_size=50_000,
                              out="data/results/anticipation_word.json",
                              ckpt="models/anticipation_word.pt"),
                 "subword": dict(window=384, stride=288, vocab_size=100_000,
                                 out="data/results/anticipation_model_subword.json",
                                 ckpt="models/anticipation_subword.pt")}
MODEL_DEFAULTS = {"scratch": dict(lr=3e-4),
                  "xlmr": dict(lr=2e-5, out="data/results/anticipation_xlmr.json",
                               ckpt="models/anticipation_xlmr.pt"),
                  "roberta": dict(lr=2e-5, path="data/subword_roberta.parquet",
                                  out="data/results/anticipation_roberta.json",
                                  ckpt="models/anticipation_roberta.pt"),
                  "mbert": dict(lr=2e-5, path="data/subword_mbert.parquet",
                                out="data/results/anticipation_mbert.json",
                                ckpt="models/anticipation_mbert.pt")}
PRETRAINED = ("xlmr", "roberta", "mbert")    # each reads its own tokenizer's pieces
PAIRS = ["Cantonese", "Arabic", "Philippines", "German", "Chinese", "Vietnamese",
         "Malay", "Japanese", "Hindi", "Korean", "Spanish", "French", "Russian",
         "Italian"]
PAIR_INDEX = {p: i for i, p in enumerate(PAIRS)}


def resolve_defaults(args):
    """Fill model- and unit-dependent defaults; no run overwrites another model's files."""
    model = getattr(args, "model", "scratch")
    if getattr(args, "dur_classes", 3) == 2 and getattr(args, "dur_head", "softmax") == "ordinal":
        raise SystemExit("--dur-classes 2 has a single cut: use --dur-head softmax")
    if model in PRETRAINED and args.unit != "subword":
        raise SystemExit(f"--model {model} reads sub-word pieces: use --unit subword")
    context = getattr(args, "context", None)
    explicit = {k for k in ("out", "ckpt") if getattr(args, k, None) is not None}
    pretrained_only = [f for f, on in (("--pair-emb", getattr(args, "pair_emb", False)),
                                       ("--lora-r", getattr(args, "lora_r", 0))) if on]
    if pretrained_only and model not in PRETRAINED:
        raise SystemExit(f"{', '.join(pretrained_only)} apply to the pretrained encoders only")
    if getattr(args, "lora_r", 0) and getattr(args, "lr", None) is None:
        args.lr = 2e-4                           # adapters start at zero and need a larger step
    for k, v in {**UNIT_DEFAULTS[args.unit], **MODEL_DEFAULTS[model]}.items():
        if getattr(args, k, None) is None:
            setattr(args, k, v)
    if args.path is None:
        args.path = DEFAULT_PATHS[args.unit]
    if getattr(args, "eval_path", None) is None:
        args.eval_path = args.path               # val (tau, curve) and test read this file
    if args.unit == "subword" and args.names != "script_words":
        # data/subword.parquet was built with the default names policy baked in.
        raise SystemExit("--names only applies to --unit word: the sub-word labels were "
                         "built with script_words (use kaggle/load_labels.ipynb for others)")
    subword_only = [f for f, on in (("--turn-sep", getattr(args, "turn_sep", False)),
                                    ("--prev-lang", getattr(args, "prev_lang", False)),
                                    ("--predict-at word_end",
                                     getattr(args, "predict_at", "piece") == "word_end"),
                                    ("--decide-at gap", getattr(args, "decide_at", "word") == "gap")) if on]
    if subword_only and args.unit != "subword":
        raise SystemExit(f"{', '.join(subword_only)} need the sub-word labels: use --unit subword")
    if context is not None:
        if model not in PRETRAINED:
            raise SystemExit("--context applies to the pretrained encoders only")
        # Each of these gives the current-token model information beyond piece t:
        # a separator or the previous word's language is context, and scoring only
        # word ends tells it that the next piece starts a new word.
        leaks = [f for f, on in (("--turn-sep", getattr(args, "turn_sep", False)),
                                 ("--prev-lang", getattr(args, "prev_lang", False)),
                                 ("--predict-at word_end",
                                  getattr(args, "predict_at", "piece") == "word_end")) if on]
        if leaks:
            raise SystemExit(f"--context 1 reads piece t only: {', '.join(leaks)} would add context")
        for k in ("out", "ckpt"):                # no run overwrites the full-prefix model
            if k not in explicit:
                root, ext = os.path.splitext(getattr(args, k))
                setattr(args, k, f"{root}_ctx{context}{ext}")
    if args.window > 512 and model in PRETRAINED:
        raise SystemExit("pretrained encoders read at most 512 positions: use --window 512 or less")
    return args


def device_of(name: str) -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("mps" if torch.backends.mps.is_available() else "cpu")


def curve_point(model, loader, dev, n_batches: int, step: int) -> dict:
    """Validation snapshot during training (``--eval-every``), on a fixed val slice.

    The slice is the first ``n_batches`` of the val split, in file order, so
    every snapshot scores the same positions. The F1 threshold is tuned on that
    same slice, which is optimistic but equally so at every step: use the curve's
    shape, not its level.

    Starting a DataLoader iterator draws one number from torch's global RNG,
    which would shift every later dropout mask. ``fork_rng`` restores the RNG
    state afterwards, so a seeded run gives the same model with or without
    ``--eval-every``.
    """
    with torch.random.fork_rng(devices=[dev] if dev.type == "cuda" else []):
        res = evaluate(model, loader, dev, n_batches)
    model.train()
    logit, y = res["logit"].astype(np.float64), res["y"].astype(np.float64)
    loss = float(np.mean(np.logaddexp(0, logit) - y * logit))     # BCE from logits
    tau, f1 = tune_threshold(1 / (1 + np.exp(-logit)), res["y"])
    dur = (duration_scores(res["dtrue"], res["dpred"], res["dprob"].shape[1]) if len(res["dtrue"])
           else {"macro_f1": 0.0})
    # Duration loss: unweighted cross-entropy (3 or 2 classes) at the dur_mask positions, the same
    # quantity as L_dur in training with --dur-class-weights none (for an ordinal head,
    # the NLL of its derived class probabilities). None if the slice has no such position.
    nd = len(res["dtrue"])
    p_true = res["dprob"][np.arange(nd), res["dtrue"].astype(int)] if nd else None
    dur_loss = float(-np.mean(np.log(np.clip(p_true, 1e-12, None)))) if nd else None
    return dict(step=step, val_switch_loss=loss, val_f1=f1, tau=tau,
                val_dur_macro_f1=dur["macro_f1"], val_dur_loss=dur_loss,
                n=int(len(y)), n_dur=int(nd))


def make_scheduler(opt, lrs, steps):
    """One-cycle schedule; the warm-up is at least 2 steps (OneCycleLR divides by it)."""
    if steps < 4:
        return torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0)
    return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lrs, total_steps=steps,
                                               pct_start=min(0.5, max(0.1, 2.5 / steps)))


def optimizer_groups(model, args):
    """``(param groups, max lrs)``. A pretrained body learns slowly and the new heads fast;
    ``--freeze-backbone`` trains the heads only (a linear probe of the pretrained features)."""
    if args.model not in PRETRAINED:
        if args.freeze_backbone:
            raise SystemExit("--freeze-backbone needs a pretrained --model")
        return [dict(params=list(model.parameters()), lr=args.lr)], [args.lr]
    if getattr(args, "lora_r", 0) and args.freeze_backbone:
        raise SystemExit("--lora-r and --freeze-backbone are exclusive: LoRA already freezes the body")
    heads = [p for n, p in model.named_parameters() if n.startswith("head_")]
    # requires_grad: with --lora-r the pretrained weights are frozen and only the
    # adapters (and the new language / pair embeddings) train at --lr.
    body = [p for n, p in model.named_parameters() if not n.startswith("head_") and p.requires_grad]
    if args.freeze_backbone:
        for p in body:
            p.requires_grad_(False)
        return [dict(params=heads, lr=args.head_lr)], [args.head_lr]
    return [dict(params=body, lr=args.lr), dict(params=heads, lr=args.head_lr)], [args.lr, args.head_lr]


DURATION_KEYS = ("duration", "duration_end_to_end", "duration_per_pair", "duration_per_direction")


def tuned_duration(res, tau, offsets, val_macro_f1) -> dict:
    """The duration report again, after ``offsets`` (fit on val) move the decision."""
    rep = score(dict(res, dpred=apply_class_offsets(res["dprob"], offsets)), tau)
    return dict(offsets=np.asarray(offsets).tolist(), val_macro_f1=float(val_macro_f1),
                report={k: rep[k] for k in DURATION_KEYS if k in rep})


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["scratch", *PRETRAINED], default="scratch")
    ap.add_argument("--backbone", default=None, help="override the Hugging Face checkpoint")
    ap.add_argument("--path", default=None)
    ap.add_argument("--eval-path", default=None,
                    help="val/test parquet (default --path); e.g. train on a SwitchLingua+LinCE mix "
                         "(src/lince.mix) but tune tau and score test on data/subword.parquet")
    ap.add_argument("--build-vocab", action="store_true")
    ap.add_argument("--vocab-size", type=int, default=None)
    ap.add_argument("--steps", type=int, default=4000)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--window", type=int, default=None)
    ap.add_argument("--stride", type=int, default=None)
    ap.add_argument("--d-model", type=int, default=256)
    ap.add_argument("--layers", type=int, default=4)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--lr", type=float, default=None, help="default: 3e-4 scratch, 2e-5 pretrained")
    ap.add_argument("--head-lr", type=float, default=1e-3, help="pretrained: learning rate of the heads")
    ap.add_argument("--dur-weight", type=float, default=0.3)
    ap.add_argument("--workers", type=int, default=0)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--eval-batches", type=int, default=0, help="0 = score the whole split")
    ap.add_argument("--train-langs", default="",
                    help="comma-separated first_languages to TRAIN on (default: all). "
                         "Everything else becomes the zero-shot set.")
    ap.add_argument("--no-pair", action="store_true",
                    help="drop the language-pair embedding (scratch model)")
    ap.add_argument("--unit", choices=["word", "subword"], default="word")
    ap.add_argument("--no-lang", action="store_true",
                    help="no language input (implied by --unit subword)")
    ap.add_argument("--out", default=None, help="default depends on --model and --unit")
    ap.add_argument("--ckpt", default=None, help="default depends on --model and --unit")
    ap.add_argument("--seed", type=int, default=None, help="seed initialisation and data order")
    ap.add_argument("--save-preds", default=None,
                    help="save val/test predictions (.npz) for offline threshold tuning")
    ap.add_argument("--dur-class-weights", choices=["none", "balanced"], default="none",
                    help="balanced: inverse-frequency weights from the train split")
    ap.add_argument("--turn-sep", action="store_true", help="separator piece between turns (pieces)")
    ap.add_argument("--prev-lang", action="store_true",
                    help="input: the language of the last finished word (pieces)")
    ap.add_argument("--predict-at", choices=["piece", "word_end"], default="piece",
                    help="word_end: train and score only at the last piece of each word")
    ap.add_argument("--decide-at", choices=["word", "gap"], default="word",
                    help="gap: put each label on the punctuation after its word, so the model has "
                         "read it (pieces; see anticipation.gap_landing)")
    ap.add_argument("--dur-head", choices=["softmax", "ordinal"], default="softmax",
                    help="ordinal: cumulative cuts P(>Small), P(>Medium) instead of a 3-way softmax")
    ap.add_argument("--dur-classes", type=int, choices=[3, 2], default=3,
                    help="2: Short (1-6) vs Large (7+) duration head, trained and scored on two "
                         "classes (default 3: Small / Medium / Large)")
    ap.add_argument("--context", type=int, choices=[1], default=None,
                    help="pretrained: 1 = read the current piece only (no prefix); default the whole prefix")
    ap.add_argument("--reshuffle-epochs", action="store_true",
                    help="a new data order every epoch (off: each epoch repeats the first's order)")
    ap.add_argument("--eval-every", type=int, default=0,
                    help="score a fixed val slice every N steps (learning curve in the json)")
    ap.add_argument("--eval-every-batches", type=int, default=200,
                    help="size of that val slice, in batches")
    ap.add_argument("--pair-emb", action="store_true",
                    help="pretrained: add a learned embedding of the row's language pair (15 pairs)")
    ap.add_argument("--lora-r", type=int, default=0,
                    help="pretrained: freeze the encoder, train rank-r LoRA adapters (lr default 2e-4)")
    ap.add_argument("--lora-alpha", type=float, default=None, help="LoRA scale numerator (default 2r)")
    ap.add_argument("--freeze-backbone", action="store_true",
                    help="pretrained: train the two heads only, the backbone stays as pretrained "
                         "(with --steps 0 nothing is trained: random heads, val-tuned decisions)")
    ap.add_argument("--names", choices=["script_words", "transparent", "script", "hybrid", "language"],
                    default="script_words",
                    help="how named entities count (see src/segments.py); word units only")
    args = ap.parse_args()
    import src.anticipation as _ant
    _ant.NAMES = args.names
    resolve_defaults(args)
    if args.seed is not None:
        set_seed(args.seed)
    use_lang = args.prev_lang or not (args.no_lang or args.unit == "subword")

    if args.model in PRETRAINED:
        vocab, vocab_size = None, None                # raw piece ids; size from the backbone
    else:
        vocab_path = VOCAB_PATHS[args.unit]
        if args.build_vocab or not os.path.exists(vocab_path):
            t0 = time.time()
            print("building vocabulary from the train split ...", flush=True)
            builder = build_vocab if args.unit == "word" else build_subword_vocab
            vocab = builder(args.path, args.vocab_size)
            os.makedirs(os.path.dirname(vocab_path), exist_ok=True)
            json.dump(vocab, open(vocab_path, "w"))
            print(f"  {len(vocab):,} types in {time.time()-t0:.0f}s -> {vocab_path}")
        vocab = json.load(open(vocab_path))
        vocab_size = len(vocab)


    train_langs = [x.strip() for x in args.train_langs.split(",") if x.strip()]
    holdout = [p for p in PAIRS if p not in train_langs] if train_langs else []
    if holdout:
        print(f"train languages : {', '.join(train_langs)}")
        print(f"zero-shot held out: {', '.join(holdout)}")

    dev = device_of(args.device)
    model = build_model(vars(args), vocab_size, n_pairs=len(PAIRS),
                        use_pair=args.model == "scratch" and not (args.no_pair or bool(holdout)),
                        use_lang=use_lang, n_dur=args.dur_classes)
    vocab_size = vocab_size or model.vocab_size
    print(f"model {args.model} | unit {args.unit} | {args.path} | vocab {vocab_size:,} | "
          f"language input {'on' if use_lang else 'off'}")
    n_param = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"device {dev} | {n_param/1e6:.1f}M params, {n_train/1e6:.1f}M trainable")

    chk = check_causality(model, vocab_size, seq_len=32, device="cpu")
    print(f"causality check: max logit drift {chk['max_abs_logit_drift']:.2e} "
          f"over {chk['positions_tested']} prefixes -> "
          f"{'PREFIX-ONLY OK' if chk['causal'] else 'LEAK'}")
    if not chk["causal"]:
        raise SystemExit("aborting: model is not causal")
    if args.context == 1:                              # passes the check above trivially
        cf = check_context_free(model, vocab_size, seq_len=32, device="cpu")
        print(f"context-free check: max logit drift {cf['max_abs_logit_drift']:.2e} when every "
              f"other position changes -> {'CURRENT PIECE ONLY OK' if cf['context_free'] else 'LEAK'}")
        if not cf["context_free"]:
            raise SystemExit("aborting: the current-token model reads other positions")
        chk["context_free"] = cf
    model.to(dev)

    sep_id = None
    if args.turn_sep and args.model in PRETRAINED:     # the backbone's own separator token
        from transformers import AutoTokenizer
        from src.xlmr_model import BACKBONES
        sep_id = AutoTokenizer.from_pretrained(args.backbone or BACKBONES[args.model]).sep_token_id
    blank_ids = None
    if args.decide_at == "gap":                        # the tokenizer's bare space markers
        from src.subword import blank_piece_ids
        from src.xlmr_model import BACKBONES
        blank_ids = blank_piece_ids(
            (args.backbone or BACKBONES[args.model]) if args.model in PRETRAINED else BACKBONES["xlmr"])
        print(f"decide_at=gap: {len(blank_ids)} blank piece ids {blank_ids[:8]}")
    common = dict(window=args.window, stride=args.stride, pair_index=PAIR_INDEX, unit=args.unit,
                  predict_at=args.predict_at, turn_sep=args.turn_sep, sep_id=sep_id,
                  decide_at=args.decide_at, blank_ids=blank_ids,
                  lang_input="prev_word" if args.prev_lang else "current",
                  seed=args.seed if args.seed is not None else 0)
    train_kw = dict(reshuffle=args.reshuffle_epochs)
    print(f"flags: seed={args.seed} predict_at={args.predict_at} decide_at={args.decide_at} turn_sep={args.turn_sep} "
          f"prev_lang={args.prev_lang} dur_class_weights={args.dur_class_weights} "
          f"context={args.context or 'prefix'} dur_head={args.dur_head} dur_classes={args.dur_classes} "
          f"freeze_backbone={args.freeze_backbone} pair_emb={args.pair_emb} lora_r={args.lora_r} "
          f"reshuffle_epochs={args.reshuffle_epochs}")
    loader = lambda split, langs, workers=0: make_loader(  # noqa: E731
        args.eval_path, vocab, split, args.batch_size, workers, languages=langs, **common)
    train_loader = make_loader(args.path, vocab, "train", args.batch_size, args.workers,
                               languages=train_langs or None, **common, **train_kw)
    val_loader, test_loader = loader("val", train_langs or None), loader("test", train_langs or None)
    curve_loader = val_loader                 # unshuffled: each pass restarts at the same rows

    groups, lrs = optimizer_groups(model, args)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    sched = make_scheduler(opt, lrs, args.steps)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    bce = nn.BCEWithLogitsLoss(reduction="none")
    dur_w = None
    if args.dur_class_weights == "balanced":
        counts = count_duration_classes(args.path, args.unit, train_langs or None, args.predict_at)
        if args.dur_classes == 2:                       # Short = Small + Medium
            counts = np.array([counts[:-1].sum(), counts[-1]])
        dur_w = duration_class_weights(counts)
        print("duration class counts (train):", counts.tolist(), "-> weights", np.round(dur_w, 3).tolist())
        dur_w = torch.tensor(dur_w, dtype=torch.float32, device=dev)
    ce = nn.CrossEntropyLoss(weight=dur_w, reduction="none")
    ordinal = args.dur_head == "ordinal"

    step, t0, running, curve = 0, time.time(), [], []
    model.train()
    while step < args.steps:
        for b in train_loader:
            if step >= args.steps:
                break
            with autocast(dev):
                sw, dur = model(b["word_ids"].to(dev), b["lang_ids"].to(dev),
                                pair_ids(model, b).to(dev))
            sw, dur = sw.float(), dur.float()
            m = b["sw_mask"].to(dev)
            loss = (bce(sw, b["y_sw"].to(dev).float()) * m).sum() / m.sum().clamp(min=1)
            dm = b["dur_mask"].to(dev)
            if dm.any():
                y_dur = batch_dur_labels(b, args.dur_classes).to(dev).reshape(-1)
                if ordinal:                  # one binary loss per cut, class-weighted like CE
                    dl = bce(dur.reshape(-1, N_DUR - 1), ordinal_targets(y_dur)).sum(-1)
                    if dur_w is not None:
                        dl = dl * dur_w[y_dur]
                else:
                    dl = ce(dur.reshape(-1, args.dur_classes), y_dur)
                loss = loss + args.dur_weight * ((dl * dm.reshape(-1)).sum() / dm.sum().clamp(min=1))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running.append(loss.detach().item())
            step += 1
            if step % 200 == 0:
                print(f"  step {step:>5}/{args.steps}  loss {np.mean(running[-200:]):.4f}"
                      f"  {step/(time.time()-t0):.2f} it/s", flush=True)
            if args.eval_every and step % args.eval_every == 0 and step < args.steps:
                pt = curve_point(model, curve_loader, dev, args.eval_every_batches, step)
                curve.append(dict(pt, train_loss=float(np.mean(running[-args.eval_every:]))))
                print(f"  [val @ {step}] switch loss {pt['val_switch_loss']:.4f}  F1 {pt['val_f1']:.4f}"
                      f"  duration loss {pt['val_dur_loss'] if pt['val_dur_loss'] is None else round(pt['val_dur_loss'], 4)}"
                      f"  macro-F1 {pt['val_dur_macro_f1']:.4f}  ({pt['n']:,} positions)",
                      flush=True)
    print(f"trained {step} steps in {(time.time()-t0)/60:.1f} min")

    val = evaluate(model, val_loader, dev, args.eval_batches)
    tau, vf1 = tune_threshold(1 / (1 + np.exp(-val["logit"])), val["y"])
    print(f"threshold tuned on val: tau={tau:.3f} (val F1 {vf1:.3f})")

    test = evaluate(model, test_loader, dev, args.eval_batches)
    if args.save_preds:
        os.makedirs(os.path.dirname(args.save_preds) or ".", exist_ok=True)
        save_predictions(args.save_preds, val=val, test=test)
        print(f"saved predictions -> {args.save_preds}")
    report = score(test, tau)
    print("\n" + format_report(report, f"TEST split - {args.model} ({args.unit})"))
    if "shared" in report:
        print("\n" + format_report(report["shared"], "SHARED positions (last piece of each "
                                    "non-CJK word; comparable across tokenizers)"))
    payload = dict(model=args.model, unit=args.unit, **report, tau=tau, val_f1=vf1,
                   params=n_param, causality=chk, steps=step, args=vars(args))
    if curve:                             # --eval-every; the last point is the final model
        last = curve_point(model, curve_loader, dev, args.eval_every_batches, step)
        curve.append(dict(last, train_loss=float(np.mean(running[-args.eval_every:]))))
        payload["curve"] = curve
    if len(val["dtrue"]):                 # the duration decision, re-balanced on val
        dur_off, dvf = tune_class_offsets(val["dprob"], val["dtrue"])
        payload["duration_tuned"] = tuned_duration(test, tau, dur_off, dvf)
        print("\n" + format_duration(payload["duration_tuned"]["report"],
                                      f"TEST split, duration offsets tuned on val "
                                      f"({', '.join(dur_labels(args.dur_classes))}) {np.round(dur_off, 2).tolist()}:"))
    if holdout:
        zt = evaluate(model, loader("test", holdout), dev, args.eval_batches)
        zs = score(zt, tau)
        print("\n" + format_report(zs, "ZERO-SHOT - language pairs never seen in training"))
        payload["zero_shot"] = dict(**zs, holdout=holdout)
        if "duration_tuned" in payload and len(zt["dtrue"]):
            payload["zero_shot"]["duration_tuned"] = tuned_duration(zt, tau, dur_off, dvf)

    os.makedirs(os.path.dirname(args.ckpt) or ".", exist_ok=True)
    torch.save(dict(state=model.state_dict(), args=vars(args), tau=tau,
                    vocab_size=vocab_size, use_lang=use_lang), args.ckpt)
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    json.dump(payload, open(args.out, "w"), indent=2, default=float)
    print(f"\nwrote {args.out} and {args.ckpt}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/training/eval_lince.py
#!/usr/bin/env python
"""Score a trained anticipation checkpoint on LinCE (human-tagged tweets).

This tests out-of-domain transfer. The model was trained on SwitchLingua's generated
dialogues, and here it reads real tweets in two of the same pairs:

    ./.venv/bin/python scripts/training/eval_lince.py --ckpt models/anticipation_<TAG>.pt
    ./.venv/bin/python scripts/training/eval_lince.py --ckpt ... --splits validation --pairs spaeng

The labels come from LinCE's gold word tags (``src/lince.py``), so this measures
the model alone, with no tagger error. Everything else matches the SwitchLingua
evaluation:

* the checkpoint's tokenizer, windowing, decision point (``--decide-at``) and language
  input (``--prev-lang`` reads the gold language of the last finished word);
* the same report (``evaluation.full_report``);
* the checkpoint's own threshold ``tau``, tuned on SwitchLingua val. This is the
  primary, honest number. The "oracle" block re-tunes tau on LinCE validation and
  scores LinCE train, as an upper bound on what recalibration alone could recover.

Duration is scored at true switches whose class is known (``dur_known``), in the
head's own classes. For a 3-class head it is also shown collapsed to Short (1-6) vs
Large (7+).

Writes ``data/results/lince_<TAG>.json``. ``--save-preds`` writes an ``.npz`` in the
``train_anticipation`` layout, with the split as the prefix (``validation_*``, ``train_*``).
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

from src import lince  # noqa: E402
from src.anticipation import collapse_duration, dur_labels  # noqa: E402
from src.evaluation import format_report  # noqa: E402
from src.metrics import duration_scores  # noqa: E402
from src.models import PRETRAINED, load_checkpoint  # noqa: E402
from src.streaming import make_loader  # noqa: E402
from src.training import evaluate, score, tune_threshold  # noqa: E402


def device_of(name: str):
    import torch
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("mps" if torch.backends.mps.is_available() else "cpu")


def concat(results: list[dict]) -> dict:
    keys = results[0].keys()
    out = {}
    for k in keys:
        parts = [r[k] for r in results if len(r[k])]
        out[k] = np.concatenate(parts) if parts else results[0][k]
    return out


def two_class(res: dict, n_dur: int, cut: float | None) -> dict | None:
    """Short (1-6) vs Large (7+) at the duration points; a 3-class head is collapsed.

    For a 3-class head the prediction is Large when P(large) > ``cut`` (the
    val-tuned cut from ``duration_binary_<tag>.json`` when given, else 0.5).
    """
    if not len(res["dtrue"]):
        return None
    if n_dur == 2:
        y, p = res["dtrue"].astype(int), res["dprob"].argmax(1)
    else:
        y = collapse_duration(res["dtrue"].astype(np.int64))
        p = (res["dprob"][:, 2] > (0.5 if cut is None else cut)).astype(int)
    s = duration_scores(y, p, 2)
    return dict(cut=cut, n=int(len(y)), accuracy=float(np.mean(y == p)),
                macro_f1=float(s["macro_f1"]), per_class_f1=[float(x) for x in s["per_class_f1"]],
                share_large=float(np.mean(y == 1)))


def block(res: dict, tau: float, n_dur: int, cut) -> dict:
    rep = score(res, tau)
    y = res["y"].astype(int)
    p = float(y.mean()) if len(y) else 0.0
    rep["base_rate"] = p
    rep["all_positive_f1"] = 2 * p / (1 + p) if p else 0.0     # F1 of "always switch"
    rep["n_rows"] = int(len(np.unique(res["row"])))
    tc = two_class(res, n_dur, cut)
    if tc:
        rep["duration_two_class"] = tc
    return rep


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--pairs", default="spaeng,hineng")
    ap.add_argument("--splits", default="validation,train")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--out", default=None, help="default data/results/lince_<TAG>.json")
    ap.add_argument("--save-preds", default=None, help="e.g. data/preds/lince_<TAG>.npz")
    ap.add_argument("--duration-cut", type=float, default=None,
                    help="3-class heads: P(large) cut for Short/Large (default: from "
                         "data/results/duration_binary_<TAG>.json, else 0.5)")
    ap.add_argument("--rebuild", action="store_true", help="re-tokenize the LinCE parquets")
    args = ap.parse_args()

    tag = os.path.splitext(os.path.basename(args.ckpt))[0].removeprefix("anticipation_")
    out = args.out or f"data/results/lince_{tag}.json"
    dev = device_of(args.device)
    model, ck = load_checkpoint(args.ckpt, map_location="cpu")
    model.to(dev)
    a = ck["args"]
    if a.get("model") not in PRETRAINED or a.get("unit") != "subword":
        raise SystemExit("eval_lince scores the pretrained sub-word models (xlmr / roberta / mbert)")
    if a.get("turn_sep"):
        raise SystemExit("--turn-sep runs are not supported: a tweet has one speaker turn")
    tau = float(ck["tau"])
    n_dur = int(a.get("dur_classes") or 3)
    from src.xlmr_model import BACKBONES
    backbone = a.get("backbone") or BACKBONES[a["model"]]
    blank_ids = None
    if a.get("decide_at", "word") == "gap":
        from src.subword import blank_piece_ids
        blank_ids = blank_piece_ids(backbone)
    cut = args.duration_cut
    bin_json = f"data/results/duration_binary_{tag}.json"
    if cut is None and n_dur == 3 and os.path.exists(bin_json):
        cut = float(json.load(open(bin_json))["two_class"]["cut"])
    print(f"{args.ckpt}: {a['model']} ({backbone}) | tau {tau:.3f} | {n_dur} duration classes "
          f"({', '.join(dur_labels(n_dur))}) | decide_at {a.get('decide_at', 'word')} | "
          f"prev_lang {bool(a.get('prev_lang'))} | device {dev}")

    common = dict(window=int(a["window"]), stride=int(a["stride"]), unit="subword",
                  predict_at=a.get("predict_at", "piece"), decide_at=a.get("decide_at", "word"),
                  blank_ids=blank_ids, lang_input="prev_word" if a.get("prev_lang") else "current")
    pairs = [p.strip() for p in args.pairs.split(",") if p.strip()]
    splits = [s.strip() for s in args.splits.split(",") if s.strip()]
    per_split, data = {}, {}
    for split in splits:
        parts = []
        for pair in pairs:
            path, stats = lince.build(pair, split, backbone, rebuild=args.rebuild)
            if stats:
                print(f"  built {path}: {stats}")
            t0 = time.time()
            loader = make_loader(path, None, "all", args.batch_size, 0, **common)
            res = evaluate(model, loader, dev)
            print(f"  {pair}/{split}: {len(res['y']):,} positions, {int(res['y'].sum()):,} switches "
                  f"in {time.time() - t0:.0f}s", flush=True)
            parts.append(res)
        data[split] = concat(parts)
        per_split[split] = block(data[split], tau, n_dur, cut)
        print("\n" + format_report(per_split[split], f"LinCE {split} - tau from SwitchLingua val"))
        tc = per_split[split].get("duration_two_class")
        if tc:
            print(f"  Short/Large: macro-F1 {tc['macro_f1']:.3f} | acc {tc['accuracy']:.3f} | "
                  f"n {tc['n']:,} | share Large {tc['share_large']:.2f}")

    payload = dict(ckpt=args.ckpt, tag=tag, tau=tau, n_dur=n_dur, duration_cut=cut,
                   pairs=pairs, splits=per_split, args=a)
    if len(splits) > 1:
        payload["all"] = block(concat(list(data.values())), tau, n_dur, cut)
        print("\n" + format_report(payload["all"], "LinCE, all splits - tau from SwitchLingua val"))
    if "validation" in data and "train" in data:
        v = data["validation"]
        otau, of1 = tune_threshold(1 / (1 + np.exp(-v["logit"])), v["y"])
        payload["oracle"] = dict(tau=otau, val_f1=of1, train=block(data["train"], otau, n_dur, cut))
        print("\n" + format_report(payload["oracle"]["train"],
                                   f"ORACLE: tau re-tuned on LinCE validation ({otau:.3f}), scored on LinCE train"))

    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    json.dump(payload, open(out, "w"), indent=2, default=float)
    print(f"\nwrote {out}")
    if args.save_preds:
        os.makedirs(os.path.dirname(args.save_preds) or ".", exist_ok=True)
        np.savez_compressed(args.save_preds, **{f"{s}_{k}": np.asarray(v)
                                                for s, r in data.items() for k, v in r.items()})
        print(f"saved predictions -> {args.save_preds}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## Build the labels for this tokenizer

In [ ]:
# Build this tokenizer's sub-word labels: our word labels + the original text.
import glob
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, snapshot_download
from transformers import AutoTokenizer
from src.segments import parse_utterances
from src.subword import build_frame, write_parquet
from src.xlmr_model import BACKBONES

PATH = {"xlmr": "data/subword.parquet", "roberta": "data/subword_roberta.parquet",
        "mbert": "data/subword_mbert.parquet"}[MODEL]
tagged = pd.read_parquet(hf_hub_download(LABELS_REPO, "tagged.parquet", repo_type="dataset", token=TOKEN),
                         columns=["row_id", "first_language", "second_language", "quality",
                                  "tokens", "token_lang", "is_entity"])
snap = snapshot_download("Shelton1013/SwitchLingua_text", repo_type="dataset",
                         allow_patterns=["*.csv"], token=TOKEN)
files = sorted(glob.glob(os.path.join(snap, "*.csv")), key=os.path.basename)   # same order as locally
cells = pd.concat([pd.read_csv(f, usecols=["data_generation_result"]) for f in files],
                  ignore_index=True)["data_generation_result"]
assert len(cells) == 234_172, f"unexpected corpus size {len(cells):,}"
texts = {int(r): parse_utterances(cells.iloc[int(r)]) for r in tagged.row_id}
tok = AutoTokenizer.from_pretrained(BACKBONES[MODEL])
frame, stats = build_frame(tagged, texts, tok)
frame = frame.iloc[np.random.default_rng(0).permutation(len(frame))]   # no curriculum by language
write_parquet(frame, PATH)
print(f"{PATH}: {stats['rows']:,} conversations, {stats['pieces']:,} pieces, "
      f"{stats['failed']} alignment failures, {stats['mixed_pieces']} mixed pieces")
te = frame[(frame.row_id % 10 == 9) & (frame.first_language == "Malay")]   # label-version check
print("Malay test switches:", int(sum((np.asarray(y) == 1).sum() for y in te.y_sw)),
      "(v2 labels with the lexical override = 7,207; v1 = 7,525)")
del tagged, cells, texts, frame, te
TRAIN_PATH = PATH
if MIX_LINCE:
    from src import lince
    TRAIN_PATH, mix_stats = lince.mix(PATH, BACKBONES[MODEL], MIX_LINCE, out=f"data/subword_mix_lince{MIX_LINCE}.parquet")
    print(f"{TRAIN_PATH}: {mix_stats['rows']:,} rows; LinCE = {mix_stats['lince_share_of_train_pieces']:.1%} "
          f"of training pieces; {mix_stats}")

## Train and evaluate (all 15 pairs)

In [ ]:
steps, evalb = (60, 20) if SMOKE else (STEPS, EVAL_BATCHES)
common = (f"python scripts/training/train_anticipation.py --model {MODEL} --unit subword "
          f"--steps {steps} --batch-size {BATCH} --window {WINDOW} --stride {STRIDE} "
          f"--lr {LORA_LR if LORA_R else LR} --head-lr {HEAD_LR} --eval-batches {evalb} --workers 2 "
          f"--seed {SEED} --dur-class-weights {DUR_WEIGHTS} --dur-head {DUR_HEAD} --predict-at {PREDICT_AT} "
          f"--decide-at {DECIDE_AT} --dur-weight {DUR_WEIGHT} --dur-classes {DUR_CLASSES}"
          + (" --turn-sep" if TURN_SEP else "") + (" --prev-lang" if PREV_LANG else "")
          + (" --pair-emb" if PAIR_EMB else "") + (f" --lora-r {LORA_R}" if LORA_R else "")
          + (f" --backbone {BACKBONE}" if BACKBONE else "")
          + (f" --eval-every {EVAL_EVERY}" if EVAL_EVERY and not SMOKE else "")
          + (" --reshuffle-epochs" if RESHUFFLE else "")
          + (f" --context {CONTEXT}" if CONTEXT else "")
          + (f" --path {TRAIN_PATH} --eval-path {PATH}" if MIX_LINCE else ""))
TAG = MODEL + ("L" if "large" in BACKBONE else "") + "".join(s for s, on in (("_durw", DUR_WEIGHTS != "none"), ("_ord", DUR_HEAD == "ordinal"),
                                      ("_bin", DUR_CLASSES == 2),
                                      ("_sep", TURN_SEP),
                                      ("_prev", PREV_LANG), ("_pair", PAIR_EMB),
                                      (f"_lora{LORA_R}", bool(LORA_R)), ("_wend", PREDICT_AT == "word_end"),
                                      ("_gap", DECIDE_AT == "gap"),
                                      (f"_dw{DUR_WEIGHT:g}", DUR_WEIGHT != 0.3),
                                      (f"_{STEPS // 1000}k", STEPS != 12_000), ("_rs", RESHUFFLE),
                                      (f"_ctx{CONTEXT}", bool(CONTEXT)),
                                      (f"_{len(PAIRS.split(','))}pairs", bool(PAIRS))) if on) + SUFFIX       + (f"_lince{MIX_LINCE}" if MIX_LINCE else "")
if RUN_FULL:
    cmd = (f"{common}" + (f" --train-langs {PAIRS}" if PAIRS else "") + f" --out data/results/anticipation_{TAG}.json --ckpt models/anticipation_{TAG}.pt "
           f"--save-preds data/preds/{TAG}.npz")
    print(cmd)
    get_ipython().system(f"{cmd} 2>&1 | tee logs/train_{TAG}.log")
else:
    print("RUN_FULL is off")

## Optional: zero-shot run (4 pairs held out)

In [ ]:
if ZERO_SHOT:
    langs = "Cantonese,Arabic,Philippines,Chinese,Vietnamese,Malay,Japanese,Russian,Spanish,French"
    cmd = (f"{common} --train-langs {langs} "
           f"--out data/results/anticipation_{TAG}_zeroshot.json --ckpt models/anticipation_{TAG}_zeroshot.pt")
    print(cmd)
    get_ipython().system(f"{cmd} 2>&1 | tee logs/train_{TAG}_zeroshot.log")
else:
    print("ZERO_SHOT is off")

## Results

In [ ]:
import json
from src.anticipation import dur_labels
for f in (f"data/results/anticipation_{TAG}.json", f"data/results/anticipation_{TAG}_zeroshot.json"):
    if not os.path.exists(f):
        continue
    r = json.load(open(f))
    o, d = r["overall"], r["duration"]
    sh = r.get("shared", {}).get("overall", {}).get("f1", float("nan"))
    print(f"== {f}")
    print(f"  switch F1 {o['f1']:.3f} (P {o['precision']:.3f}, R {o['recall']:.3f}) | "
          f"macro-F1 {r['macro_f1']:.3f} | shared F1 {sh:.3f}")
    print("  by direction:", {k: round(v["f1"], 3) for k, v in r["per_direction"].items()})
    names = dur_labels(len(d["per_class_f1"]))
    print(f"  duration: {len(names)}-class accuracy {d['accuracy']:.3f} | macro-F1 {d['macro_f1']:.3f}")
    if "zero_shot" in r:
        z = r["zero_shot"]
        print(f"  ZERO-SHOT held-out F1 {z['overall']['f1']:.3f} | macro {z['macro_f1']:.3f} | "
              + ", ".join(f"{k} {v['f1']:.3f}" for k, v in z["per_pair"].items()))
    display(pd.DataFrame({k: {"F1": v["f1"], "P": v["precision"], "R": v["recall"], "n": v["n"]}
                          for k, v in r["per_pair"].items()}).T.sort_values("F1", ascending=False).round(3))
    if "duration_tuned" in r:
        t = r["duration_tuned"]["report"]["duration"]
        print(f"  duration, offsets tuned on val {r['duration_tuned']['offsets']}: accuracy "
              f"{t['accuracy']:.3f} | macro-F1 {t['macro_f1']:.3f} | F1 {'/'.join(names)} "
              + " / ".join(f"{f:.3f}" for f in t["per_class_f1"]))
    if "duration_end_to_end" in r:
        e = r["duration_end_to_end"]
        print(f"  duration end to end (switch predicted AND class right): accuracy "
              f"{e['accuracy']:.3f} | macro-F1 {e['macro_f1']:.3f}")
    if "duration_per_pair" in r:
        display(pd.DataFrame({k: {"n": v["n"], "acc": v["accuracy"], "macro-F1": v["macro_f1"],
                                  **{f"F1 {c}": f for c, f in zip(names, v["per_class_f1"])},
                                  "e2e acc": v.get("end_to_end", {}).get("accuracy")}
                              for k, v in r["duration_per_pair"].items()}).T
                .sort_values("macro-F1", ascending=False).round(3))

## LinCE: out-of-domain tweets
Scores the checkpoint on LinCE Spanish-English and Hindi-English tweets (validation + train, both human-tagged and never trained on; with `MIX_LINCE`, validation only, because LinCE train was trained on). Writes `data/results/lince_<TAG>.json`.

In [ ]:
# Out-of-domain check: the trained checkpoint on LinCE tweets (human word tags, so no tagger error).
# The threshold is the one tuned on SwitchLingua val; an "oracle" block re-tunes it on LinCE val.
ck = f"models/anticipation_{TAG}.pt"
if EVAL_LINCE and RUN_FULL and os.path.exists(ck):
    cmd = (f"python scripts/training/eval_lince.py --ckpt {ck} --batch-size 64"
           + (" --splits validation --pairs hineng" if SMOKE else " --splits validation" if MIX_LINCE else "")
           + f" --out data/results/lince_{TAG}.json --save-preds data/preds/lince_{TAG}.npz")
    print(cmd)
    get_ipython().system(f"{cmd} 2>&1 | tee logs/lince_{TAG}.log")
else:
    print("LinCE evaluation skipped (EVAL_LINCE off, RUN_FULL off, or no checkpoint)")

## Clean up
Removes the derived training file (it comes from private labels). Result JSONs stay in `data/results/`.

In [ ]:
if not SAVE_CHECKPOINT:
    for f in (f"models/anticipation_{TAG}.pt", f"models/anticipation_{TAG}_zeroshot.pt"):
        if os.path.exists(f):
            os.remove(f)
for f in {PATH, TRAIN_PATH}:                  # derived from the private labels: don't publish them
    if os.path.exists(f):
        os.remove(f)
print(sorted(os.listdir("data")), sorted(os.listdir("models")))